In [ ]:
import gc
from typing import final
import multiprocessing as mp
import numpy as np
import glob2
import datetime
from datetime import timedelta, tzinfo,timezone
from matplotlib.colors import ListedColormap
import colorcet as cc
from pathlib import Path
from scipy import stats
from scipy.optimize import least_squares
import networkx as nx
from collections import defaultdict
from scipy.stats import chi2
from torch.compiler import disable
from tqdm import tqdm
import pickle
from matplotlib import pyplot as plt
import pandas as pd
from utils.data_reading.sound_data.station import StationsCatalog
from utils.physics.sound_model.spherical_sound_model import GridSphericalSoundModel as GridSoundModel
from utils.physics.sound_model.spherical_sound_model import HomogeneousSphericalSoundModel as HomogeneousGridSoundModel
from utils.detection.association_geodesic_ridges import compute_candidates, update_valid_grid, update_results, load_detections, compute_grids

# plot parameters
import matplotlib as mpl
mpl.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 12,
    "font.family": "serif",
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

In [ ]:
import os
from utils.physics.sound_model.ellipsoidal_sound_model import GridEllipsoidalSoundModel

# paths
CATALOG_PATH = "/media/rsafran/CORSAIR/OHASISBIO/recensement_stations_OHASISBIO_RS.csv"  # csv catalog files
DETECTIONS_DIR = "/media/rsafran/CORSAIR/T-pick_2.2"  # where we have detection pickles
ISAS_PATH = "/media/rsafran/CORSAIR/ISAS/extracted/2018"
YEAR = 2018
 # output dir
ASSOCIATIONS_DIR = f"../../data/detection/T-pick_2.2/{YEAR}"
Path(ASSOCIATIONS_DIR).mkdir(exist_ok=True)

# delimitation of the detections we keep (this notebook actually associates the detections of 1 year and 4 hours)
DATE_START = datetime.datetime(YEAR, 1, 1) - datetime.timedelta(hours=2)
DATE_END = datetime.datetime(YEAR+1, 3, 1) + datetime.timedelta(hours=2)

# Detections loading parameters
MIN_P_TISSNET_PRIMARY = 0.5 # min probability of browsed detections
MIN_P_TISSNET_SECONDARY = 0.1  # min probability of detections that can be associated with the browsed one
MERGE_DELTA_S = 5 # threshold below which we consider two events should be merged

# The REQ_CLOSEST_STATIONS th closest stations will be required for an association to be valid
# e.g. if we set it to 6, no association of size <6 will be saved (this is useful to save memory)
REQ_CLOSEST_STATIONS = 6

# sound model definition
arr = os.listdir(ISAS_PATH)
file_list = [os.path.join(ISAS_PATH, fname) for fname in arr if fname.endswith('.nc')]
SOUND_MODEL = GridEllipsoidalSoundModel(file_list)
# GRID_TO_COORDS is a list giving, for each cell of the grid, the coordinates of its center (e.g. GRID_TO_COORDS[25] is a (lat,lon) tuple accounting for cell 25)
with open("../../data/T-pick/grid_to_coords_ridges.pkl", "rb") as f:
    GRID_TO_COORDS = pickle.load(f)

# association running parameters
SAVE_PATH_ROOT = None  # change this to save the grids as figures, leave at None by default

STATIONS = StationsCatalog(CATALOG_PATH).filter_out_undated().filter_out_unlocated()
# load detection files and keep all the ones that are included in the wanted year
FILES = {}
for f in glob2.glob(f"{DETECTIONS_DIR}/*.pkl"):
    det_files = [f for f in glob2.glob(DETECTIONS_DIR + "/*") if Path(f).is_file()]
    det_files = [f for f in det_files if "2018" in f ]
    dataset, s_name = f[:-4].split("/")[-1].split("_")
    s = STATIONS.by_dataset(dataset).by_name(s_name)
    if len(s) != 1:
        print(f"station {dataset}_{s_name} not found or not unique")
        continue
    FILES[s[0]] = f
FILES = {s : FILES[s] for s in FILES if (s.date_end > DATE_START and s.date_start < DATE_END)}

In [ ]:
lat_min, lon_min = GRID_TO_COORDS.min(axis=0)
lat_max, lon_max = GRID_TO_COORDS.max(axis=0)
LAT_BOUNDS = (lat_min, lat_max)
LON_BOUNDS = (lon_min, lon_max)

In [ ]:
FILES_2018 = {}
for key in FILES.keys() :
    if key.dataset == "OHASISBIO-2018" or key.dataset == "IMS-2018":
        FILES_2018[key] = FILES[key]
FILES = FILES_2018

In [ ]:
# load properly the detection files
Path(f"{ASSOCIATIONS_DIR}/cache").mkdir(parents=True, exist_ok=True)
DET_PATH = f"{ASSOCIATIONS_DIR}/cache/detections_{MIN_P_TISSNET_SECONDARY}_{MERGE_DELTA_S}.pkl"
if True :#not Path(DET_PATH).exists():
    DETECTIONS = load_detections(list(FILES.values()), STATIONS, MIN_P_TISSNET_SECONDARY, merge_delta=datetime.timedelta(seconds=MERGE_DELTA_S))
    for s in DETECTIONS.keys():
        DETECTIONS[s] = DETECTIONS[s][(DETECTIONS[s][:,0] > DATE_START) & (DETECTIONS[s][:,0] < DATE_END)]
    with open(DET_PATH, "wb") as f:
        pickle.dump((DETECTIONS), f)
else:
    with open(DET_PATH, "rb") as f:
        DETECTIONS = pickle.load(f)

# do not keep detection entries for which the detection list is empty
to_del = []
for s in DETECTIONS.keys():
    if len(DETECTIONS[s]) == 0:
        to_del.append(s)
    # if s.name == 'H04S1':# or s.name == 'H04N1' or s.name == 'H01W1':
    #     to_del.append(s)
for s in to_del:
    del DETECTIONS[s]

# assign an index to each detection
idx_det = 0
IDX_TO_DET = {}
for idx, s in enumerate(DETECTIONS.keys()):
    s.idx = idx  # indexes to store efficiently the associations
    DETECTIONS[s] = list(DETECTIONS[s])
    for i in range(len(DETECTIONS[s])):
        DETECTIONS[s][i] = np.concatenate((DETECTIONS[s][i], [idx_det]))
        IDX_TO_DET[idx_det] = DETECTIONS[s][i]
        idx_det += 1
    DETECTIONS[s] = np.array(DETECTIONS[s])
DETECTION_IDXS = np.array(list(range(idx_det)))

# only keep the stations that appear in the kept detections
STATIONS = [s for s in DETECTIONS.keys()]
FIRSTS_DETECTIONS = {s : DETECTIONS[s][0,0] for s in STATIONS}
LASTS_DETECTIONS = {s : DETECTIONS[s][-1,0] for s in STATIONS}

# list that will be browsed
DETECTIONS_MERGED = np.concatenate([[(det[0], det[1], det[2], s) for det in DETECTIONS[s]] for s in STATIONS if "IMS" not in s.dataset])
DETECTIONS_MERGED = DETECTIONS_MERGED[DETECTIONS_MERGED[:, 1] > MIN_P_TISSNET_PRIMARY]
DETECTIONS_MERGED = DETECTIONS_MERGED[np.argsort(DETECTIONS_MERGED[:, 1])][::-1]

In [ ]:
PICK_UNCERTAINTY = 5
SOUND_SPEED_UNCERTAINTY = 3
MAX_CLOCK_DRIFT = 0
GRID_PATH = f"{ASSOCIATIONS_DIR}/cache/grids_{PICK_UNCERTAINTY}_{SOUND_SPEED_UNCERTAINTY}_{MAX_CLOCK_DRIFT}.pkl"
with open(GRID_PATH, "rb") as f:
    GRID_TO_COORDS, TDoA, MAX_TDoA, TDoA_UNCERTAINTIES = pickle.load(f)

In [ ]:
# Traitement terminé: 2021569 associations trouvées
files = glob2.glob(f"{DETECTIONS_DIR}/cache/associations.pkl")
with open(files[0], "rb") as f:
    associations = pickle.load(f)

In [ ]:
with open(f'{DETECTIONS_DIR}/cache/results.pkl', 'rb') as f:
    results = pickle.load(f)

In [ ]:
import math
from collections import Counter
from itertools import combinations


# Ajouter les fonctions d'évaluation de qualité
def calculate_reduced_chi_square(res):
    """Calcule le chi-carré réduit à partir du résultat de l'optimisation."""
    n_observations = len(res.fun)
    n_parameters = 2
    degrees_of_freedom = n_observations - n_parameters
    chi_square = np.sum(res.fun**2)
    reduced_chi_square = chi_square / degrees_of_freedom if degrees_of_freedom > 0 else float('inf')
    return reduced_chi_square, chi_square, degrees_of_freedom

def azimuth_deg(lat1, lon1, lat2, lon2):
    """
    Azimut (de lat1,lon1 -> lat2,lon2) en degrés [0,360),
    0 = nord, 90 = est, sens horaire.
    Utilise la formule géodésique sphérique.
    """
    lat1, lon1, lat2, lon2 = map(math.radians, (lat1, lon1, lat2, lon2))
    dlon = lon2 - lon1
    x = math.sin(dlon) * math.cos(lat2)
    y = math.cos(lat1)*math.sin(lat2) - math.sin(lat1)*math.cos(lat2)*math.cos(dlon)
    az = math.degrees(math.atan2(x, y))
    return (az + 360.0) % 360.0

def calculate_azimuthal_gap(sensors_positions, event_position, skip_same=True):
    """
    Calcule le gap azimutal maximal entre les stations.
    sensors_positions: iterable de [lat, lon] en degrés
    event_position: [lat, lon, ...] en degrés (seuls lat et lon sont utilisés)
    skip_same: si True, ignore les capteurs exactement à la position de l'événement
    Retour: gap max en degrés (0-360)
    """
    event_lat, event_lon = event_position[0], event_position[1]
    azims = []
    for s in sensors_positions:
        s_lat, s_lon = s[0], s[1]
        if skip_same and math.isclose(s_lat, event_lat, abs_tol=1e-12) and math.isclose(s_lon, event_lon, abs_tol=1e-12):
            # capteur au même endroit que l'événement -> on l'ignore (pas d'azimut utile)
            continue
        az = azimuth_deg(event_lat, event_lon, s_lat, s_lon)
        azims.append(az)

    if len(azims) == 0:
        return 360.0
    if len(azims) == 1:
        return 360.0

    azims_sorted = np.sort(np.mod(azims, 360.0))
    diffs = np.diff(azims_sorted)
    # gap circulaire entre dernier et premier
    last_gap = 360.0 - (azims_sorted[-1] - azims_sorted[0])
    gaps = np.append(diffs, last_gap)
    return float(np.max(gaps))


def estimate_position_uncertainty(res):
    """Estime l'incertitude sur la position à partir de la matrice de covariance."""
    try:
        J = res.jac
        covariance_matrix = np.linalg.inv(J.T @ J) * (2 * res.cost) / (len(res.fun) - 2)
        x_uncertainty = np.sqrt(covariance_matrix[0, 0])
        y_uncertainty = np.sqrt(covariance_matrix[1, 1])
        return (x_uncertainty, y_uncertainty), covariance_matrix
    except:
        return (float('inf'), float('inf')), None

def quality_score(res, sensors_positions, weights=None, max_chi_sq=3.0, max_gap=180.0):
    """Calcule un score de qualité combinant plusieurs indicateurs."""
    # Reduced chi-square
    red_chi_sq, _, _ = calculate_reduced_chi_square(res)
    chi_sq_score = max(0, 1 - (red_chi_sq / max_chi_sq))

    # Gap azimutal
    event_pos = (res.x[1], res.x[2]) if len(res.x) > 2 else (res.x[0], res.x[1])
    gap = calculate_azimuthal_gap(sensors_positions, event_pos)
    gap_score = max(0, 1 - (gap / max_gap))

    # Résidus standardisés
    std_residuals = res.fun
    outlier_score = 1 - (np.sum(np.abs(std_residuals) > 3.0) / len(std_residuals))

    # Score combiné (pondéré)
    combined_score = 0.5 * chi_sq_score + 0.5 * outlier_score
    return combined_score, {
        'red_chi_sq': red_chi_sq,
        'azimuthal_gap': gap,
        'chi_sq_score': chi_sq_score,
        'gap_score': gap_score,
        'outlier_score': outlier_score
    }

def filter_by_cost_threshold(results_dict, cost_threshold=10000):
    """
    Filtre les associations par seuil de coût.

    Args:
        results_dict: Dictionnaire des résultats d'optimisation
        cost_threshold: Seuil de coût maximum

    Returns:
        dict: results_dict filtré par coût
    """
    print(f"Step 1: Filtering by cost threshold ({cost_threshold})...")

    filtered_results = {}
    for i, res in results_dict.items():
        cost = res.cost if hasattr(res, 'cost') else np.sum(res.fun**2)
        if cost < cost_threshold:
            filtered_results[i] = res

    print(f"Results after cost filtering: {len(filtered_results)}")
    return filtered_results


def remove_subset_associations(results_dict, associations, keep_best_duplicate=True):
    """
    Supprime les associations qui utilisent un sous-ensemble de détections
    par rapport à d'autres associations.
    Version optimisée.
    """
    print("Step 2: Removing subset associations...")

    # Préparer les données avec les sets de détections
    events_data = []
    for i in results_dict.keys():
        detections = associations[i][0][:, 1]  # Index des détections
        det_set = frozenset(detections)
        # cost = results_dict[i].cost if hasattr(results_dict[i], 'cost') else np.sum(results_dict[i].fun**2)
        cost = np.mean(results_dict[i].fun**2)
        date = datetime.datetime.fromtimestamp(results_dict[i].x[0],tz=timezone.utc).date()
        events_data.append({
            'uid': i,
            "date": date,
            'detection_set': det_set,
            # 'cost': cost,
            'set_size': len(det_set),
            'cost' : cost / len(det_set)

        })

    # Trier par taille décroissante, puis par coût croissant
    print("Sorting by detection set size (descending) and cost (ascending)...")
    events_data.sort(key=lambda x: (x['cost'], -x['set_size']))

    # Structures optimisées
    accepted_by_date = defaultdict(list)  # Acceptés par date
    accepted_by_size = defaultdict(list)   # Acceptés par taille pour accès rapide
    excluded_indices = set()
    seen_detection_sets = {}  # dict pour éviter recherches répétées

    for current in tqdm(events_data, desc="Filtering subsets"):
        current_set = current['detection_set']
        current_size = current['set_size']
        date = current['date']
        is_subset = False

        # Vérification rapide des doublons exacts
        if current_set in seen_detection_sets:
            if keep_best_duplicate:
                excluded_indices.add(current['uid'])
                continue
            # Si on ne garde pas le meilleur, vérifier si current est meilleur
            elif current['cost'] < seen_detection_sets[current_set]['cost']:
                # Exclure l'ancien, garder le nouveau
                excluded_indices.add(seen_detection_sets[current_set]['uid'])
                excluded_indices.remove(current['uid']) if current['uid'] in excluded_indices else None
                is_subset = False
            else:
                excluded_indices.add(current['uid'])
                continue

        # Récupérer les événements des dates voisines (optimisé)
        dates_to_check = [date - timedelta(days=1), date, date + timedelta(days=1)]

        # Ne vérifier que les ensembles de taille >= current_size
        # Car un sous-ensemble strict a toujours une taille inférieure
        for check_date in dates_to_check:
            if check_date not in accepted_by_date:
                continue

            for accepted in accepted_by_date[check_date]:
                # Optimisation: skip si l'ensemble accepté est plus petit
                if accepted['set_size'] < current_size:
                    continue

                # Optimisation: vérification rapide avec len avant issubset
                if current_size <= accepted['set_size']:
                    if current_set.issubset(accepted['detection_set']):
                        # C'est un sous-ensemble (strict ou égal)
                        if current_set != accepted['detection_set'] or keep_best_duplicate:
                            excluded_indices.add(current['uid'])
                            is_subset = True
                            break

            if is_subset:
                break

        if not is_subset:
            accepted_by_date[date].append(current)
            accepted_by_size[current_size].append(current)
            seen_detection_sets[current_set] = current

    print(f"Events after subset removal: {sum(len(v) for v in accepted_by_date.values())}")
    print(f"Excluded subset events: {len(excluded_indices)}")

    # Créer le dictionnaire filtré
    accepted_events = [x for xs in accepted_by_date.values() for x in xs]
    filtered_results = {event['uid']: results_dict[event['uid']] for event in accepted_events}

    return filtered_results, excluded_indices


from src.utils.physics.sound_model.quality_tester import LocalizationQualityTester, QualityLevel, QualityReport

def remove_overlap_bipartite(results_dict, association):

    tester = LocalizationQualityTester(n_params=2)


    """Version avec graphe biparti (memory-efficient)"""
    print("Step 3: Filtrage avec structure bipartite...")
    def compute_quality_metrics(result):
        """Utilise le QualityTester complet (test global + Baarda/Pope + score)"""

        report = tester.evaluate(result)

        # Cas trop faible
        if report.level == 0:  # QualityLevel.REJECTED
            return {
                'quality_score': -np.inf,
                'passes_chi2': False,
                'num_obs': report.n_obs,
                'chi2_reduced': report.sigma2_hat,
                'outlier_fraction': len(report.outlier_indices) / max(report.n_obs,1),
                'max_residual_zscore': max(report.outlier_scores) if report.outlier_scores else 0,
                'cost': np.inf,
                'selection_score': report.selection_score,
                'level': report.level
            }

        return {
            'quality_score': report.selection_score,     # Score robuste
            'passes_chi2': report.global_test_passed,
            'num_obs': report.n_obs,
            'chi2_reduced': report.sigma2_hat,
            'outlier_fraction': len(report.outlier_indices) / report.n_obs,
            'max_residual_zscore': max(report.outlier_scores) if report.outlier_scores else 0,
            'cost': report.details['global']['vtpv'],
            'selection_score': report.selection_score,
            'level': report.level
        }

    # === PHASE 1 : Construire structure implicite ===
    events = {}
    det_to_events = defaultdict(set)  # Garder ce mapping !

    for i in tqdm(results_dict.keys(), desc="Building bipartite structure"):
        try:
            arr = np.asarray(association[i][0])
            detections = arr[:, 1].astype(int) if arr.ndim == 2 and arr.size > 0 else []
        except Exception:
            detections = []

        det_set = frozenset(detections)
        quality_metrics = compute_quality_metrics(results_dict[i])

        events[i] = {
            'detection_set': det_set,
            'num_stations': arr.shape[0] if arr.ndim == 2 else 0,
            'quality': quality_metrics
        }

        for det in det_set:
            det_to_events[det].add(i)

    print(f"Bipartite graph: {len(events)} events, {len(det_to_events)} detections")

    # === PHASE 2 : Trouver conflits SANS créer le graphe ===
    # Union-Find sur les événements directement
    parent = {i: i for i in events.keys()}

    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        parent[find(x)] = find(y)

    # Parcourir les arêtes implicites du graphe biparti
    print("Finding connected components via implicit edges...")
    for det, event_set in tqdm(det_to_events.items(), desc="Processing detections"):
        if len(event_set) > 1:
            event_list = list(event_set)
            # Union tous les événements partageant cette détection
            for i in range(1, len(event_list)):
                union(event_list[0], event_list[i])

    # === PHASE 3 : Grouper par composante ===
    clusters_dict = defaultdict(set)
    for event_id in events:
        clusters_dict[find(event_id)].add(event_id)

    clusters = [c for c in clusters_dict.values()]

    isolated_events = [list(c)[0] for c in clusters if len(c) == 1]
    overlapping_clusters = [c for c in clusters if len(c) > 1]

    print(f"Found {len(isolated_events)} isolated, {len(overlapping_clusters)} clusters")

    def event_priority_score(event_id):
        ev = events[event_id]
        qual = ev['quality']
        # Rejet des mauvais niveaux
        if qual['level'].name == "REJECTED" or qual['level'].name == "POOR":
            return (-np.inf, 0, 0, 0)
        # Rejet immédiat
        if qual['num_obs'] < 3:
            return (-np.inf, 0, 0, 0)
        if not qual['passes_chi2']:
            return (-np.inf, 0, 0, 0)

        # Le score principal = sélection robuste du QualityTester
        main_score = qual['selection_score']

        return (
            main_score,
            -qual['chi2_reduced'],
            -qual['outlier_fraction'],
            ev['num_stations']
        )

    selected_events = set()
    selected_detections = set()

    for cluster in tqdm(overlapping_clusters, desc="Resolving clusters"):
        cluster_events = sorted(cluster, key=event_priority_score, reverse=True)
        cluster_selected = []
        cluster_used_dets = set()

        for event_id in cluster_events:
            event_dets = events[event_id]['detection_set']

            if (event_dets.isdisjoint(cluster_used_dets) and
                event_priority_score(event_id)[0] > -np.inf):
                cluster_selected.append(event_id)
                cluster_used_dets.update(event_dets)

        if not cluster_selected and cluster_events:
            best = cluster_events[0]
            if event_priority_score(best)[0] > -np.inf:
                cluster_selected = [best]
                cluster_used_dets = events[best]['detection_set']

        selected_events.update(cluster_selected)
        selected_detections.update(cluster_used_dets)

    # Ajouter isolés
    for event_id in isolated_events:
        if (event_priority_score(event_id)[0] > -np.inf and
            events[event_id]['detection_set'].isdisjoint(selected_detections)):
            selected_events.add(event_id)
            selected_detections.update(events[event_id]['detection_set'])

    final_events = list(selected_events)

    # === Statistiques ===
    if final_events:
        quality_scores = [events[eid]['quality']['quality_score'] for eid in final_events]
        print(f"\n=== RESULTS ===")
        print(f"Selected {len(final_events)}/{len(events)} events")
        print(f"Quality: {np.mean(quality_scores):.3f} ± {np.std(quality_scores):.3f}")

    return {eid: results_dict[eid] for eid in final_events if eid in results_dict}


def fallback_selection(events, used_detections):
    """Sélection de repli si la sélection principale échoue"""
    candidates = []
    for event_id, ev_data in events.items():
        if (ev_data['quality']['num_obs'] >= 3 and
            ev_data['detection_set'].isdisjoint(used_detections)):
            candidates.append(event_id)

    # Tri par qualité
    candidates.sort(key=lambda x: events[x]['quality']['quality_score'], reverse=True)

    selected = []
    current_dets = set(used_detections)

    for cand in candidates:
        cand_dets = events[cand]['detection_set']
        if current_dets.isdisjoint(cand_dets):
            selected.append(cand)
            current_dets.update(cand_dets)

    return selected

def remove_overlap_bipartite(results_dict, association,
                                       n_params=2,
                                       alpha_global=0.05,
                                       alpha_outlier=0.001,
                                       min_quality=QualityLevel.MARGINAL,
                                       bilateral_test=False,
                                       stations=None,
                                       gap_threshold=270.0):
    """
    Filtrage des localisations avec tests statistiques rigoureux

    Args:
        results_dict: Dict {event_id: OptimizeResult}
        association: Dict {event_id: (array_associations, ...)}
        n_params: Nombre de paramètres (2 pour x,y)
        alpha_global: Niveau du test global (défaut 5%)
        alpha_outlier: Niveau pour outliers individuels (défaut 0.1%)
        min_quality: Niveau minimum acceptable (défaut MARGINAL)
        bilateral_test: False = unilatéral (rejette seulement σ² >> 1)
        stations: Dict/list des stations pour calcul azimuthal gap (optionnel)
        gap_threshold: Seuil max d'azimuthal gap en degrés (défaut 270°)
    """
    print("=== Filtrage avec tests statistiques rigoureux ===")
    print(f"    Test global: {'bilatéral' if bilateral_test else 'unilatéral (σ² > 1 seulement)'}")
    if stations is not None:
        print(f"    Azimuthal gap: activé (seuil = {gap_threshold}°)")

    # Initialiser le testeur
    tester = LocalizationQualityTester(
        n_params=n_params,
        alpha_global=alpha_global,
        alpha_outlier=alpha_outlier,
        bilateral_test=bilateral_test
    )

    # === PHASE 1 : Évaluer toutes les localisations ===
    print("\n[1/4] Évaluation de la qualité des localisations...")
    events = {}
    det_to_events = defaultdict(set)

    quality_stats = {level: 0 for level in QualityLevel}

    for event_id in tqdm(results_dict.keys(), desc="Quality evaluation"):
        result = results_dict[event_id]

        # Extraire les détections associées
        try:
            arr = np.asarray(association[event_id][0])
            detections = arr[:, 1].astype(int) if arr.ndim == 2 and arr.size > 0 else []
        except Exception:
            detections = []

        det_set = frozenset(detections)

        # Évaluation rigoureuse
        quality_report = tester.evaluate(result)
        quality_stats[quality_report.level] += 1

        events[event_id] = {
            'detection_set': det_set,
            'quality': quality_report,
            'n_stations': arr.shape[0] if arr.ndim == 2 else 0
        }

        for det in det_set:
            det_to_events[det].add(event_id)

    # Afficher statistiques de qualité
    print("\nDistribution des niveaux de qualité:")
    for level in QualityLevel:
        count = quality_stats[level]
        pct = 100 * count / len(events) if events else 0
        print(f"  {level.name:12s}: {count:5d} ({pct:5.1f}%)")

    # === PHASE 2 : Pré-filtrage par qualité minimale ===
    print(f"\n[2/4] Pré-filtrage (qualité >= {min_quality.name})...")

    valid_events = {
        eid for eid, ev in events.items()
        if ev['quality'].level.value >= min_quality.value
    }

    rejected_quality = len(events) - len(valid_events)
    print(f"  Rejetés pour qualité insuffisante: {rejected_quality}")

    # === PHASE 3 : Trouver composantes connexes ===
    print("\n[3/4] Identification des conflits...")

    # Union-Find
    parent = {i: i for i in valid_events}

    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        parent[find(x)] = find(y)

    for det, event_set in det_to_events.items():
        valid_in_set = [e for e in event_set if e in valid_events]
        if len(valid_in_set) > 1:
            for i in range(1, len(valid_in_set)):
                union(valid_in_set[0], valid_in_set[i])

    # Grouper
    clusters_dict = defaultdict(set)
    for event_id in valid_events:
        clusters_dict[find(event_id)].add(event_id)

    clusters = list(clusters_dict.values())
    isolated = [list(c)[0] for c in clusters if len(c) == 1]
    conflicting = [c for c in clusters if len(c) > 1]

    print(f"  Événements isolés: {len(isolated)}")
    print(f"  Clusters conflictuels: {len(conflicting)}")

    # === PHASE 4 : Résolution des conflits ===
    print("\n[4/4] Résolution des conflits par sélection optimale...")

    def selection_key(event_id):
        """
        Clé de tri pour sélection (plus grand = meilleur)

        Priorité :
        1. Niveau de qualité (EXCELLENT > GOOD > ...)
        2. Test global passé
        3. Nombre d'observations (plus = mieux, plus robuste)
        4. σ² proche de 1 mais sans pénaliser σ² < 1 excessivement
        """
        ev = events[event_id]
        q = ev['quality']

        # Pour σ², pénaliser seulement > 1 (outliers/sous-estimation erreurs)
        # σ² < 1 est acceptable (surestimation conservatrice des erreurs)
        sigma2 = q.sigma2_hat if np.isfinite(q.sigma2_hat) else 100.0
        sigma2_penalty = max(0, sigma2 - 1.0)  # 0 si σ² <= 1, positif sinon

        return (
            q.level.value,                    # Niveau de qualité
            1 if q.global_test_passed else 0, # Test global
            q.n_obs,                          # Nombre d'observations
            -len(q.outlier_indices),          # Moins d'outliers = mieux
            -sigma2_penalty,                  # Pénalité σ² > 1 seulement
            ev['n_stations']                  # Tie-breaker
        )

    selected_events = set()
    selected_detections = set()
    selection_reasons = defaultdict(list)

    # Traiter les clusters conflictuels
    for cluster in tqdm(conflicting, desc="Resolving conflicts"):
        # Trier par qualité décroissante
        sorted_events = sorted(cluster, key=selection_key, reverse=True)

        cluster_selected = []
        cluster_used_dets = set()

        for event_id in sorted_events:
            event_dets = events[event_id]['detection_set']

            # Sélectionner si pas de conflit avec déjà sélectionnés
            if event_dets.isdisjoint(cluster_used_dets):
                cluster_selected.append(event_id)
                cluster_used_dets.update(event_dets)
                selection_reasons['selected_in_cluster'].append(event_id)
            else:
                # Conflit - vérifier si remplacement justifié
                overlap = event_dets & cluster_used_dets
                selection_reasons['rejected_overlap'].append(
                    (event_id, len(overlap))
                )

        selected_events.update(cluster_selected)
        selected_detections.update(cluster_used_dets)

    # Ajouter les événements isolés
    for event_id in isolated:
        event_dets = events[event_id]['detection_set']
        if event_dets.isdisjoint(selected_detections):
            selected_events.add(event_id)
            selected_detections.update(event_dets)
            selection_reasons['isolated'].append(event_id)

    # === RÉSUMÉ ===
    print("\n" + "="*50)
    print("RÉSUMÉ DE LA SÉLECTION")
    print("="*50)
    print(f"Événements en entrée:     {len(results_dict)}")
    print(f"Rejetés (qualité):        {rejected_quality}")
    print(f"Événements sélectionnés:  {len(selected_events)}")
    print(f"Détections utilisées:     {len(selected_detections)}")

    # Statistiques sur les sélectionnés
    if selected_events:
        selected_qualities = [events[e]['quality'] for e in selected_events]

        sigma2_values = [q.sigma2_hat for q in selected_qualities if np.isfinite(q.sigma2_hat)]
        scores = [q.selection_score for q in selected_qualities if np.isfinite(q.selection_score)]

        print(f"\nQualité des sélectionnés:")
        print(f"  σ² moyen: {np.mean(sigma2_values):.3f} ± {np.std(sigma2_values):.3f}")
        print(f"  Score moyen: {np.mean(scores):.3f} ± {np.std(scores):.3f}")

        level_counts = defaultdict(int)
        for q in selected_qualities:
            level_counts[q.level] += 1

        print(f"\n  Distribution:")
        for level in QualityLevel:
            if level_counts[level] > 0:
                print(f"    {level.name}: {level_counts[level]}")

    # Conserver la structure de sortie originale : {event_id: OptimizeResult}
    return {eid: results_dict[eid] for eid in selected_events if eid in results_dict}

def remove_overlap_bipartite_cost_only(results_dict, association):
    """
    Version simplifiée de remove_overlap_bipartite utilisant uniquement le coût pour résoudre les conflits.
    """
    print("Step 3: Filtrage avec structure bipartite (basé uniquement sur le coût)...")

    def compute_cost(result):
        """Calcule le coût pour un résultat donné."""
        return np.sum(result.fun**2)

    # === PHASE 1 : Construire structure implicite ===
    events = {}
    det_to_events = defaultdict(set)  # Garder ce mapping !

    for i in tqdm(results_dict.keys(), desc="Building bipartite structure"):
        try:
            arr = np.asarray(association[i][0])
            detections = arr[:, 1].astype(int) if arr.ndim == 2 and arr.size > 0 else []
        except Exception:
            detections = []

        det_set = frozenset(detections)
        cost = compute_cost(results_dict[i])

        events[i] = {
            'detection_set': det_set,
            'cost': cost
        }

        for det in det_set:
            det_to_events[det].add(i)

    print(f"Bipartite graph: {len(events)} events, {len(det_to_events)} detections")

    # === PHASE 2 : Trouver conflits SANS créer le graphe ===
    # Union-Find sur les événements directement
    parent = {i: i for i in events.keys()}

    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        parent[find(x)] = find(y)

    # Parcourir les arêtes implicites du graphe biparti
    print("Finding connected components via implicit edges...")
    for det, event_set in tqdm(det_to_events.items(), desc="Processing detections"):
        if len(event_set) > 1:
            event_list = list(event_set)
            # Union tous les événements partageant cette détection
            for i in range(1, len(event_list)):
                union(event_list[0], event_list[i])

    # === PHASE 3 : Grouper par composante ===
    clusters_dict = defaultdict(set)
    for event_id in events:
        clusters_dict[find(event_id)].add(event_id)

    clusters = [c for c in clusters_dict.values()]

    isolated_events = [list(c)[0] for c in clusters if len(c) == 1]
    overlapping_clusters = [c for c in clusters if len(c) > 1]

    print(f"Found {len(isolated_events)} isolated, {len(overlapping_clusters)} clusters")

    def event_priority_score(event_id):
        """Clé de tri basée uniquement sur le coût."""
        return -events[event_id]['cost']  # Plus le coût est bas, meilleur est le score

    selected_events = set()
    selected_detections = set()

    for cluster in tqdm(overlapping_clusters, desc="Resolving clusters"):
        cluster_events = sorted(cluster, key=event_priority_score, reverse=True)
        cluster_selected = []
        cluster_used_dets = set()

        for event_id in cluster_events:
            event_dets = events[event_id]['detection_set']

            if event_dets.isdisjoint(cluster_used_dets):
                cluster_selected.append(event_id)
                cluster_used_dets.update(event_dets)

        if not cluster_selected and cluster_events:
            best = cluster_events[0]
            cluster_selected = [best]
            cluster_used_dets = events[best]['detection_set']

        selected_events.update(cluster_selected)
        selected_detections.update(cluster_used_dets)

    # Ajouter isolés
    for event_id in isolated_events:
        if events[event_id]['detection_set'].isdisjoint(selected_detections):
            selected_events.add(event_id)
            selected_detections.update(events[event_id]['detection_set'])

    final_events = list(selected_events)

    # === Statistiques ===
    if final_events:
        costs = [events[eid]['cost'] for eid in final_events]
        print(f"\n=== RESULTS ===")
        print(f"Selected {len(final_events)}/{len(events)} events")
        print(f"Mean cost: {np.mean(costs):.3f} ± {np.std(costs):.3f}")

    return {eid: results_dict[eid] for eid in final_events if eid in results_dict}

def remove_overlap_bipartite_chi2(results_dict, association):
    """
    Version simplifiée de remove_overlap_bipartite utilisant uniquement le reduced chi-square pour résoudre les conflits.
    """
    print("Step 3: Filtrage avec structure bipartite (basé uniquement sur le reduced chi-square)...")

    def compute_reduced_chi_square(result):
        """Calcule le reduced chi-square pour un résultat donné."""
        n_observations = len(result.fun)
        n_parameters = 2  # Supposons que nous avons 2 paramètres (par exemple, x et y)
        degrees_of_freedom = n_observations - n_parameters
        chi_square = np.sum(result.fun**2)
        reduced_chi_square = chi_square / degrees_of_freedom if degrees_of_freedom > 0 else float('inf')
        return reduced_chi_square

    # === PHASE 1 : Construire structure implicite ===
    events = {}
    det_to_events = defaultdict(set)  # Garder ce mapping !

    for i in tqdm(results_dict.keys(), desc="Building bipartite structure"):
        try:
            arr = np.asarray(association[i][0])
            detections = arr[:, 1].astype(int) if arr.ndim == 2 and arr.size > 0 else []
        except Exception:
            detections = []

        det_set = frozenset(detections)
        reduced_chi_square = compute_reduced_chi_square(results_dict[i])

        events[i] = {
            'detection_set': det_set,
            'reduced_chi_square': reduced_chi_square
        }

        for det in det_set:
            det_to_events[det].add(i)

    print(f"Bipartite graph: {len(events)} events, {len(det_to_events)} detections")

    # === PHASE 2 : Trouver conflits SANS créer le graphe ===
    # Union-Find sur les événements directement
    parent = {i: i for i in events.keys()}

    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(x, y):
        parent[find(x)] = find(parent[y])

    # Parcourir les arêtes implicites du graphe biparti
    print("Finding connected components via implicit edges...")
    for det, event_set in tqdm(det_to_events.items(), desc="Processing detections"):
        if len(event_set) > 1:
            event_list = list(event_set)
            # Union tous les événements partageant cette détection
            for i in range(1, len(event_list)):
                union(event_list[0], event_list[i])

    # === PHASE 3 : Grouper par composante ===
    clusters_dict = defaultdict(set)
    for event_id in events:
        clusters_dict[find(event_id)].add(event_id)

    clusters = [c for c in clusters_dict.values()]

    isolated_events = [list(c)[0] for c in clusters if len(c) == 1]
    overlapping_clusters = [c for c in clusters if len(c) > 1]

    print(f"Found {len(isolated_events)} isolated, {len(overlapping_clusters)} clusters")

    def event_priority_score(event_id):
        """Clé de tri basée uniquement sur le reduced chi-square."""
        return -events[event_id]['reduced_chi_square']  # Plus le reduced chi-square est bas, meilleur est le score

    selected_events = set()
    selected_detections = set()

    for cluster in tqdm(overlapping_clusters, desc="Resolving clusters"):
        cluster_events = sorted(cluster, key=event_priority_score, reverse=True)
        cluster_selected = []
        cluster_used_dets = set()

        for event_id in cluster_events:
            event_dets = events[event_id]['detection_set']

            if event_dets.isdisjoint(cluster_used_dets):
                cluster_selected.append(event_id)
                cluster_used_dets.update(event_dets)

        if not cluster_selected and cluster_events:
            best = cluster_events[0]
            cluster_selected = [best]
            cluster_used_dets = events[best]['detection_set']

        selected_events.update(cluster_selected)
        selected_detections.update(cluster_used_dets)

    # Ajouter isolés
    for event_id in isolated_events:
        if events[event_id]['detection_set'].isdisjoint(selected_detections):
            selected_events.add(event_id)
            selected_detections.update(events[event_id]['detection_set'])

    final_events = list(selected_events)

    # === Statistiques ===
    if final_events:
        reduced_chi_squares = [events[eid]['reduced_chi_square'] for eid in final_events]
        print(f"\n=== RESULTS ===")
        print(f"Selected {len(final_events)}/{len(events)} events")
        print(f"Mean reduced chi-square: {np.mean(reduced_chi_squares):.3f} ± {np.std(reduced_chi_squares):.3f}")

    return {eid: results_dict[eid] for eid in final_events if eid in results_dict}



def get_quality_diagnostics(results_dict, association, n_params=2,
                            alpha_global=0.01, alpha_outlier=0.0001):
    """
    Génère un diagnostic de qualité pour tous les événements
    (fonction utilitaire séparée pour l'analyse)

    Args:
        results_dict: Dict {event_id: OptimizeResult}
        association: Dict {event_id: (array_associations, ...)}

    Returns:
        DataFrame avec diagnostics détaillés
    """
    import pandas as pd

    tester = LocalizationQualityTester(n_params, alpha_global, alpha_outlier)

    rows = []
    for eid, result in results_dict.items():
        q = tester.evaluate(result)

        # Nombre de détections
        try:
            arr = np.asarray(association[eid][0])
            n_det = arr.shape[0] if arr.ndim == 2 else 0
        except:
            n_det = 0

        rows.append({
            'event_id': eid,
            'quality_level': q.level.name,
            'sigma2_hat': q.sigma2_hat,
            'global_test_passed': q.global_test_passed,
            'n_outliers': len(q.outlier_indices),
            'outlier_indices': q.outlier_indices,
            'n_obs': q.n_obs,
            'dof': q.dof,
            'selection_score': q.selection_score,
            'chi2_lower': q.chi2_bounds[0],
            'chi2_upper': q.chi2_bounds[1],
            'n_detections': n_det
        })

    return pd.DataFrame(rows)

def filter_associations_simple(results_dict, associations, cost_threshold=10000):
    """
    Version améliorée du filtrage des associations avec fonctions séparées.

    Args:
        results_dict: Dictionnaire des résultats d'optimisation
        associations: Structure contenant les associations de détections
        cost_threshold: Seuil de coût pour le filtrage initial

    Returns:
        dict: Dictionnaire des résultats filtrés
    """
    print("Starting associations filtering...")
    print(f"Initial associations count: {len(results_dict)}")

    # Étape 1: Filtrer par seuil de coût
    filtered_by_cost = filter_by_cost_threshold(results_dict, cost_threshold)
    del results_dict
    # filtered_by_cost = results_dict
    # Étape 2: Supprimer les associations subset
    # filtered_no_subsets, excluded_indices = remove_subset_associations(filtered_by_cost, associations)
    #
    filtered_no_subsets =filtered_by_cost
    excluded_indices = []
    del filtered_by_cost

    # Étape 3: Résoudre les overlaps
    final_results = remove_overlap_bipartite(filtered_no_subsets, associations)

    # final_results = filtered_no_subsets
    del filtered_no_subsets
    print("associations filtering completed!")
    print(f"Final results: {len(final_results)} associations")

    return final_results, excluded_indices

def run_filtered_associations():

    # Étape 2: Filtrer et déduplicater

    # with open(f"{DETECTIONS_DIR}/cache/results.pkl", "rb") as f:
    with open('../../data/localisation/results_iter2.pkl', "rb") as f:
        results = pickle.load(f)

    filtered_results, excluded_indices = filter_associations_simple(results,associations, cost_threshold=100)

    # Étape 3: Sauvegarder
    # Sauvegarder juste les résultats comme array (compatible avec votre format original)
    results_array = [filtered_results[i] for i in sorted(filtered_results.keys())]
    np.save("filtered_results.npy", np.array(results_array))
    with open("filtered_results.pkl", "wb") as f:
        pickle.dump(filtered_results, f)
    # Optionnel: sauvegarder les métadonnées
    metadata = {
        'filtered_indices': list(filtered_results.keys()),
        'excluded_indices': list(excluded_indices),
        'original_count': len(associations),
        'processed_count': len(results),
        'final_count': len(filtered_results)
    }
    np.save("association_metadata.npy", metadata)

    print(f"Processing complete:")
    print(f"  Original associations: {len(associations)}")
    print(f"  Successfully processed: {len(results)}")
    print(f"  After filtering: {len(filtered_results)}")
    print(f"  Excluded: {len(excluded_indices)}")

    return filtered_results

# Utilisation
if __name__ == "__main__":
    import sys
    print(sys.getrecursionlimit())
    sys.setrecursionlimit(50000)
    final_results = run_filtered_associations()


In [ ]:
def load_ridge_data(dorsal_db_path):
    """
    Charge les données des dorsales océaniques
    """
    dorsal_files = [f for f in os.listdir(dorsal_db_path) if f.endswith('.xy')]
    print(f"Loading {len(dorsal_files)} ridge files: {dorsal_files}")

    ridge_data = {}
    all_ridge_points = []

    for f in dorsal_files:
        ridge_name = f.replace('axe-', '').replace('.xy', '')
        df = pd.read_csv(os.path.join(dorsal_db_path, f),
                         comment=">", sep=r'\s+')

        ridge_points = df[['y', 'x']].values

        ridge_data[ridge_name] = ridge_points
        all_ridge_points.append(ridge_points)

        print(f"  {ridge_name}: {len(ridge_points)} points")

    # Combinaison de toutes les dorsales
    all_ridge_points = np.vstack(all_ridge_points)
    print(f"Total ridge points: {len(all_ridge_points)}")

    return ridge_data, all_ridge_points

ridge_data_path="../../data/dorsales/"
_ ,ridge_points = load_ridge_data(ridge_data_path)

In [ ]:
import numpy as np
from numpy.linalg import norm

def distance(p1: np.ndarray, p2: np.ndarray) -> float:
    """Calcule la distance euclidienne entre deux points."""
    return norm(p1 - p2)

def find_closest(p3: np.ndarray, ridge_points: np.ndarray) -> tuple[int, int]:
    """
    Trouve les deux points les plus proches de p3 dans ridge_points.

    Args:
        p3: Point dont on veut trouver les plus proches voisins.
        ridge_points: Tableau de points représentant la dorsale.

    Returns:
        Tuple d'indices des deux points les plus proches.
    """
    # Calcule les distances entre p3 et tous les points de la dorsale
    distances = np.array([distance(p3, p) for p in ridge_points])

    # Trouve l'indice du point le plus proche
    place1 = np.argmin(distances)

    # Détermine l'indice du deuxième point (voisin gauche ou droit)
    if place1 == 0:
        place2 = place1 + 1
    elif place1 == len(ridge_points) - 1:
        place2 = place1 - 1
    else:
        # Compare les distances des voisins gauche et droit
        if distances[place1 - 1] < distances[place1 + 1]:
            place2 = place1 - 1
        else:
            place2 = place1 + 1

    return place1, place2

# def find_min_distance(p3: np.ndarray, place1: int, place2: int, ridge_points: np.ndarray) -> float:
#     """
#     Calcule la distance minimale entre p3 et le segment défini par place1 et place2.
#
#     Args:
#         p3: Point dont on veut calculer la distance.
#         place1: Indice du premier point du segment.
#         place2: Indice du deuxième point du segment.
#         ridge_points: Tableau de points représentant la dorsale.
#
#     Returns:
#         Distance minimale entre p3 et le segment.
#     """
#     p1, p2 = ridge_points[place1], ridge_points[place2]
#
#     # Calcule la distance entre p3 et le segment p1-p2
#     segment_vec = p2 - p1
#     point_vec = p1 - p3
#
#     # Projection du vecteur point_vec sur segment_vec
#     segment_length_sq = np.dot(segment_vec, segment_vec)
#     if segment_length_sq == 0:  # p1 et p2 sont le même point
#         return distance(p3, p1)
#
#     t = max(0, min(1, np.dot(point_vec, segment_vec) / segment_length_sq))
#     projection = p1 + t * segment_vec
#
#     # Distance entre p3 et sa projection sur le segment
#     return distance(p3, projection)


from pyproj import Geod
import numpy as np

geod = Geod(ellps="WGS84")  # Utilise l'ellipsoïde WGS84

def project_point_to_segment(p1, p2, p3):
    """
    Trouve le point sur le segment [p1, p2] le plus proche de p3.

    Args:
        p1, p2: Coordonnées des extrémités du segment (lat, lon).
        p3: Coordonnées du point à projeter (lat, lon).

    Returns:
        Coordonnées du point projeté (lat, lon).
    """
    lat1, lon1 = p1
    lat2, lon2 = p2
    lat3, lon3 = p3

    # Convertit les coordonnées en radians pour les calculs
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)
    lat3_rad, lon3_rad = np.radians(lat3), np.radians(lon3)

    # Vecteur du segment [p1, p2] en coordonnées cartesiennes locales
    # On utilise une approximation plane pour la projection
    # (car la projection orthogonale exacte sur une sphère est complexe)
    # On travaille en coordonnées cartesiennes locales (x, y) :
    # x = longitude, y = latitude (approximation locale)
    x1, y1 = lon1, lat1
    x2, y2 = lon2, lat2
    x3, y3 = lon3, lat3

    # Vecteur du segment
    vx, vy = x2 - x1, y2 - y1
    # Vecteur de p1 à p3
    wx, wy = x3 - x1, y3 - y1

    # Produit scalaire v.v et v.w
    v_dot_v = vx**2 + vy**2
    v_dot_w = vx * wx + vy * wy

    # Paramètre de projection
    t = max(0, min(1, v_dot_w / v_dot_v)) if v_dot_v != 0 else 0

    # Coordonnées du point projeté
    proj_x = x1 + t * vx
    proj_y = y1 + t * vy

    return (proj_y, proj_x)  # Retourne (lat, lon)

def find_min_distance(p3, place1, place2, ridge_points):
    """
    Calcule la distance minimale entre p3 et le segment [place1, place2].

    Args:
        p3: Point dont on veut calculer la distance (lat, lon).
        place1, place2: Indices des extrémités du segment.
        ridge_points: Tableau de points de la dorsale.

    Returns:
        Distance minimale en kilomètres.
    """
    p1 = ridge_points[place1]
    p2 = ridge_points[place2]

    # Trouve le point projeté sur le segment
    proj_lat, proj_lon = project_point_to_segment(p1, p2, p3)

    # Calcule la distance géodésique entre p3 et le point projeté
    _, _, distance = geod.inv(p3[1], p3[0], proj_lon, proj_lat)

    return distance / 1000  # Convertit en kilomètres


In [ ]:
final_results

In [ ]:
# Extraire les positions des événements
event_positions = []
dist_from_ridge_ch2 = []
costs = []
is_valid = []

for i, res in final_results.items():
    if hasattr(res, 'x') and len(res.x) >= 2:
        # Supposer que res.x = [t0, lat, lon] ou [lat, lon]
        if len(res.x) == 3:
            lat, lon = res.x[1], res.x[2]
        else:
            lat, lon = res.x[0], res.x[1]
        event_positions.append([lat, lon])
        p3=[lat, lon]
        # Trouve les deux points les plus proches
        place1, place2 = find_closest(p3, ridge_points)
        # Calcule la distance minimale
        min_distance = find_min_distance(p3, place1, place2, ridge_points)
        dist_from_ridge_ch2.append(min_distance)
        cost = res.cost if hasattr(res, 'cost') else np.sum(res.fun**2)
        costs.append(cost)

In [ ]:
# # plt.hist(np.array(dist_from_ridge_raw), bins=50,density=True,log=True,alpha=0.5,edgecolor='gray')
# plt.hist(np.array(dist_from_ridge_cost), bins=50,density=True,log=True,alpha=0.5,edgecolor='gray', label='cost')
# plt.hist(np.array(dist_from_ridge_ch2), bins=50,density=True,log=True,alpha=0.5,edgecolor='gray', label='chi2')
# plt.hist(np.array(dist_from_ridge), bins=50,density=True,log=True,alpha=0.5,edgecolor='gray',label = 'Multi-critearia')
# plt.xlabel('distance from ridge axis in degree')
# plt.legend()

In [ ]:
from pygments.lexers import go


def plot_association_map(results, stations_dict, title="Associations localisées"):
    """
    Carte montrant les positions des événements et des stations
    """
    fig, ax = plt.subplots(figsize=(12, 8))

    # Extraire les positions des événements
    event_positions = []
    costs = []
    is_valid = []

    for i, res in results.items():
        if hasattr(res, 'x') and len(res.x) >= 2:
            # Supposer que res.x = [t0, lat, lon] ou [lat, lon]
            if len(res.x) == 3:
                lat, lon = res.x[1], res.x[2]
            else:
                lat, lon = res.x[0], res.x[1]
            event_positions.append([lat, lon])
            cost = res.cost if hasattr(res, 'cost') else np.sum(res.fun**2)
            costs.append(cost)

    if event_positions:
        event_positions = np.array(event_positions)

        costs = np.array(costs)

        # Scatter plot des événements colorés par cost
        scatter = ax.scatter(event_positions[:, 1], event_positions[:, 0],
                           c=costs, cmap='viridis_r', s=5, alpha=0.7,
                           label='Événements')
        plt.colorbar(scatter, label='Cost')

        # Ajouter les stations si disponibles
        if stations_dict:
            station_lats = []
            station_lons = []
            for station in stations_dict:
                pos = station.get_pos()
                station_lats.append(pos[0])
                station_lons.append(pos[1])

            ax.scatter(station_lons, station_lats, c='red', marker='^',
                      s=100, label='Stations', alpha=0.8)

    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

def plot_cost_distribution(results, bins=50):
    """
    Distribution des costs
    """
    costs = []
    red_chi_squares = []

    for i, res in results.items():
        cost = res.cost if hasattr(res, 'cost') else np.sum(res.fun**2)
        costs.append(cost)

        # Calculer reduced chi-square
        n_obs = len(res.fun)
        n_params = 2
        dof = n_obs - n_params
        red_chi_sq = cost / dof if dof > 0 else float('inf')
        red_chi_squares.append(red_chi_sq)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Distribution des costs
    ax1.hist(costs, bins=bins, alpha=0.7, edgecolor='black')
    ax1.set_xlabel('Cost')
    ax1.set_ylabel('Nombre d\'associations')
    ax1.set_title('Distribution des Costs')
    ax1.axvline(np.median(costs), color='red', linestyle='--',
                label=f'Médiane: {np.median(costs):.1f}')
    ax1.legend()

    # Distribution des reduced chi-square
    red_chi_squares = [x for x in red_chi_squares if x < 100]  # Filtrer les valeurs extrêmes
    ax2.hist(red_chi_squares, bins=bins, alpha=0.7, edgecolor='black')
    ax2.set_xlabel('Reduced Chi-square')
    ax2.set_ylabel('Nombre d\'associations')
    ax2.set_title('Distribution des Reduced Chi-square')
    ax2.axvline(1.0, color='green', linestyle='--', label='Valeur idéale')
    ax2.axvline(np.median(red_chi_squares), color='red', linestyle='--',
                label=f'Médiane: {np.median(red_chi_squares):.2f}')
    ax2.legend()

    plt.tight_layout()
    return fig, costs, red_chi_squares

def plot_residuals_analysis(results, association_data, idx_to_det, stations_dict):
    """
    Analyse des résidus par station
    """
    station_residuals = {}

    for i, res in results.items():
        stations_idx = association_data[i][0][:,0]
        residuals = res.fun

        for j, station_idx in enumerate(stations_idx):
            if j < len(residuals):
                if station_idx not in station_residuals:
                    station_residuals[station_idx] = []
                station_residuals[station_idx].append(residuals[j])

    # Box plot des résidus par station
    fig, ax = plt.subplots(figsize=(12, 6))

    station_names = []
    residual_data = []

    for station_idx, residuals in station_residuals.items():
        if len(residuals) >= 5:  # Au moins 5 mesures
            station_names.append(f'Station {station_idx}')
            residual_data.append(residuals)

    if residual_data:
        ax.boxplot(residual_data, labels=station_names)
        ax.set_ylabel('Résidus (s)')
        ax.set_title('Distribution des résidus par station')
        ax.grid(True, alpha=0.3)
        plt.xticks(rotation=45)

    plt.tight_layout()
    return fig

def plot_quality_metrics(results, association_data, stations_dict):
    """
    Métriques de qualité des associations
    """
    quality_data = []

    for i, res in results.items():
        try:
            if True :
                # Récupérer les positions des stations
                stations_idx = association_data[i][0][:,0]
                stations_pos = [stations_dict[idx].get_pos() for idx in stations_idx]

                # Calculer les métriques de base
                cost = res.cost if hasattr(res, 'cost') else np.sum(res.fun**2)

                # Reduced chi-square
                n_obs = len(res.fun)
                n_params = 2
                dof = n_obs - n_params
                red_chi_sq = cost / dof if dof > 0 else float('inf')

                # Gap azimutal
                event_pos = (res.x[1], res.x[2]) if len(res.x) > 2 else (res.x[0], res.x[1])

                # Calculer azimuts
                azimuths = []
                for sensor_pos in stations_pos:
                    dx = sensor_pos[1] - event_pos[1]
                    dy = sensor_pos[0] - event_pos[0]
                    azimuth = np.degrees(np.arctan2(dx, dy)) % 360
                    azimuths.append(azimuth)

                # Calculer gap azimutal
                azimuths_sorted = sorted(azimuths)
                gaps = []
                for j in range(len(azimuths_sorted)):
                    next_idx = (j + 1) % len(azimuths_sorted)
                    gap = (azimuths_sorted[next_idx] - azimuths_sorted[j]) % 360
                    gaps.append(gap)
                az_gap = max(gaps) if gaps else 360

                # Incertitude position (approximation simple)
                try:
                    if hasattr(res, 'jac') and res.jac is not None:
                        J = res.jac
                        cov_matrix = np.linalg.inv(J.T @ J) * (2 * cost) / max(dof, 1)
                        max_uncert = np.sqrt(max(np.diag(cov_matrix)))
                    else:
                        max_uncert = None
                except:
                    max_uncert = None

                quality_data.append({
                    'association_id': i,
                    'cost': cost,
                    'reduced_chi_sq': red_chi_sq,
                    'azimuthal_gap': az_gap,
                    'max_uncertainty': max_uncert,
                    'num_stations': len(stations_pos)
                })
        except Exception as e:
            print(f"Erreur pour l'associations {i}: {e}")
            continue

    df = pd.DataFrame(quality_data)

    if df.empty:
        print("Aucune donnée de qualité disponible")
        return plt.figure(), pd.DataFrame()

    # Créer une figure avec subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # Cost vs Reduced Chi-square
    if 'cost' in df.columns and 'reduced_chi_sq' in df.columns:
        valid_data = df[(df['reduced_chi_sq'] != float('inf')) & (df['reduced_chi_sq'] < 100)]
        if not valid_data.empty:
            axes[0,0].scatter(valid_data['cost'], valid_data['reduced_chi_sq'], alpha=0.6)
            axes[0,0].set_xlabel('Cost')
            axes[0,0].set_ylabel('Reduced Chi-square')
            axes[0,0].set_title('Cost vs Reduced Chi-square')
            axes[0,0].grid(True, alpha=0.3)

    # Gap azimutal vs Nombre de stations
    if 'num_stations' in df.columns and 'azimuthal_gap' in df.columns:
        axes[0,1].scatter(df['num_stations'], df['azimuthal_gap'], alpha=0.6)
        axes[0,1].set_xlabel('Nombre de stations')
        axes[0,1].set_ylabel('Gap azimutal (°)')
        axes[0,1].set_title('Gap azimutal vs Nombre de stations')
        axes[0,1].grid(True, alpha=0.3)

    # Incertitude vs Cost
    if 'max_uncertainty' in df.columns and 'cost' in df.columns:
        df_clean = df.dropna(subset=['max_uncertainty'])
        df_clean = df_clean[df_clean['max_uncertainty'] != float('inf')]
        if not df_clean.empty:
            axes[1,0].scatter(df_clean['cost'], df_clean['max_uncertainty'], alpha=0.6)
            axes[1,0].set_xlabel('Cost')
            axes[1,0].set_ylabel('Incertitude max position')
            axes[1,0].set_title('Incertitude vs Cost')
            axes[1,0].grid(True, alpha=0.3)
        else:
            axes[1,0].text(0.5, 0.5, 'Pas de données d\'incertitude disponibles',
                          ha='center', va='center', transform=axes[1,0].transAxes)

    # Histogramme du gap azimutal
    if 'azimuthal_gap' in df.columns:
        axes[1,1].hist(df['azimuthal_gap'], bins=30, alpha=0.7, edgecolor='black')
        axes[1,1].axvline(180, color='red', linestyle='--', label='Seuil critique (180°)')
        axes[1,1].set_xlabel('Gap azimutal (°)')
        axes[1,1].set_ylabel('Nombre d\'associations')
        axes[1,1].set_title('Distribution du gap azimutal')
        axes[1,1].legend()

    plt.tight_layout()
    return fig, df

def create_interactive_map(results, stations_dict):
    """
    Carte interactive avec Plotly
    """
    # Préparer les données
    event_data = []

    for i, res in results.items():
        if hasattr(res, 'x') and len(res.x) >= 2:
            if len(res.x) == 3:
                lat, lon = res.x[1], res.x[2]
            else:
                lat, lon = res.x[0], res.x[1]

            cost = res.cost if hasattr(res, 'cost') else np.sum(res.fun**2)

            event_data.append({
                'lat': lat,
                'lon': lon,
                'cost': cost,
                'association_id': i,
                'type': 'Événement'
            })

    # Ajouter les stations
    station_data = []
    if stations_dict:
        for idx, station in enumerate(stations_dict):
            pos = station.get_pos()
            station_data.append({
                'lat': pos[0],
                'lon': pos[1],
                'station_id': idx,
                'type': 'Station'
            })

    # Créer la figure Plotly
    fig = go.Figure()

    # Ajouter les événements
    if event_data:
        events_df = pd.DataFrame(event_data)
        fig.add_trace(go.Scatter(
            x=events_df['lon'],
            y=events_df['lat'],
            mode='markers',
            marker=dict(
                size=8,
                color=events_df['cost'],
                colorscale='Viridis_r',
                colorbar=dict(title="Cost"),
                showscale=True
            ),
            text=events_df.apply(lambda row: f"associations {row['association_id']}<br>Cost: {row['cost']:.1f}", axis=1),
            hovertemplate='%{text}<br>Lat: %{y:.4f}<br>Lon: %{x:.4f}<extra></extra>',
            name='Événements'
        ))

    # Ajouter les stations
    if station_data:
        stations_df = pd.DataFrame(station_data)
        fig.add_trace(go.Scatter(
            x=stations_df['lon'],
            y=stations_df['lat'],
            mode='markers',
            marker=dict(
                size=12,
                color='red',
                symbol='triangle-up'
            ),
            text=stations_df.apply(lambda row: f"Station {row['station_id']}", axis=1),
            hovertemplate='%{text}<br>Lat: %{y:.4f}<br>Lon: %{x:.4f}<extra></extra>',
            name='Stations'
        ))

    fig.update_layout(
        title="Carte interactive des associations",
        xaxis_title="Longitude",
        yaxis_title="Latitude",
        showlegend=True,
        height=600
    )

    return fig

def generate_summary_report(results, association_data, stations_dict):
    """
    Génère un rapport de synthèse
    """
    print("="*60)
    print("RAPPORT DE SYNTHÈSE DES ASSOCIATIONS")
    print("="*60)

    # Statistiques générales
    print(f"\n📊 STATISTIQUES GÉNÉRALES")
    print(f"   • Nombre total d'associations: {len(results)}")

    # Analyse des costs
    costs = [res.cost if hasattr(res, 'cost') else np.sum(res.fun**2) for res in results.values()]
    print(f"\n💰 ANALYSE DES COSTS")
    print(f"   • Cost moyen: {np.mean(costs):.2f}")
    print(f"   • Cost médian: {np.median(costs):.2f}")
    print(f"   • Cost min/max: {np.min(costs):.2f} / {np.max(costs):.2f}")
    print(f"   • Écart-type: {np.std(costs):.2f}")

    # Seuils de qualité
    good_threshold = np.percentile(costs, 75)
    excellent_threshold = np.percentile(costs, 25)

    excellent = sum(1 for c in costs if c <= excellent_threshold)
    good = sum(1 for c in costs if excellent_threshold < c <= good_threshold)
    poor = sum(1 for c in costs if c > good_threshold)

    print(f"\n⭐ QUALITÉ DES ASSOCIATIONS")
    print(f"   • Excellentes (Q1 : < {excellent_threshold:.1f}): {excellent} ({excellent/len(costs)*100:.1f}%)")
    print(f"   • Bonnes (Q2-Q3 : < {good_threshold:.1f}): {good} ({good/len(costs)*100:.1f}%)")
    print(f"   • Moyennes (Q4 : > {good_threshold:.1f}): {poor} ({poor/len(costs)*100:.1f}%)")

    # Analyse des reduced chi-square
    red_chi_sqs = []
    for res in results.values():
        n_obs = len(res.fun)
        n_params = 2
        dof = n_obs - n_params
        red_chi_sq = (res.cost if hasattr(res, 'cost') else np.sum(res.fun**2)) / dof if dof > 0 else float('inf')
        if red_chi_sq != float('inf'):
            red_chi_sqs.append(red_chi_sq)

    if red_chi_sqs:
        print(f"\n📈 REDUCED CHI-SQUARE")
        print(f"   • Médian: {np.median(red_chi_sqs):.2f}")
        print(f"   • < 1.5 (bon ajustement): {sum(1 for x in red_chi_sqs if x < 1.5)} ({sum(1 for x in red_chi_sqs if x < 1.5)/len(red_chi_sqs)*100:.1f}%)")
        print(f"   • > 3.0 (mauvais ajustement): {sum(1 for x in red_chi_sqs if x > 3.0)} ({sum(1 for x in red_chi_sqs if x > 3.0)/len(red_chi_sqs)*100:.1f}%)")

# Fonction principale de visualisation
def visualize_association_results(results, association_data, stations_dict, idx_to_det,
                                 save_plots=True, output_dir="./plots/"):
    """
    Lance toutes les visualisations
    """

    if save_plots:
        os.makedirs(output_dir, exist_ok=True)

    print("Génération des visualisations...")

    # 1. Rapport de synthèse
    generate_summary_report(results, association_data, stations_dict)

    # 2. Carte des associations
    fig1 = plot_association_map(results, stations_dict)
    if save_plots:
        fig1.savefig(f"{output_dir}/association_map.png", dpi=300, bbox_inches='tight')

    # 3. Distribution des costs
    fig2, costs, red_chi_sqs = plot_cost_distribution(results)
    if save_plots:
        fig2.savefig(f"{output_dir}/cost_distribution.png", dpi=300, bbox_inches='tight')

    # 4. Analyse des résidus
    fig3 = plot_residuals_analysis(results, association_data, idx_to_det, stations_dict)
    if save_plots:
        fig3.savefig(f"{output_dir}/residuals_analysis.png", dpi=300, bbox_inches='tight')

    # 5. Métriques de qualité
    fig4, quality_df = plot_quality_metrics(results, association_data, stations_dict)
    if save_plots:
        fig4.savefig(f"{output_dir}/quality_metrics.png", dpi=300, bbox_inches='tight')

    # # 6. Carte interactive (Plotly)
    # fig_interactive = create_interactive_map(results, stations_dict)
    # if True:
    #     fig_interactive.write_html(f"{output_dir}/interactive_map.html")
    #
    # plt.show()

    return {
        'costs': costs,
        'red_chi_squares': red_chi_sqs,
        'quality_dataframe': quality_df,
        # 'interactive_map': fig_interactive
    }

# Utilisation
if __name__ == "__main__":
    # Supposons que vous avez vos résultats filtrés
    # results = {...}  # Votre dictionnaire de résultats

    # Lancer toutes les visualisations
    viz_results = visualize_association_results(
        results=final_results,
        association_data=associations,
        stations_dict=STATIONS,
        idx_to_det=IDX_TO_DET,
        save_plots=True,
        output_dir="./association_plots/"
    )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

def compute_ellipsoide_error(res):
    # 1 : Calcul de la matrice de covariance
    if hasattr(res, 'jac') and res.jac is not None:
        J = res.jac
        n_obs = len(res.fun)  # Nombre d'observations
        n_params = 2  # latitude et longitude
        dof = n_obs - n_params  # Degrés de liberté
        if dof > 0:
            covariance_matrix = np.linalg.inv(J.T @ J) * (res.cost / dof)
        else:
            covariance_matrix = np.linalg.inv(J.T @ J)
    else:
        raise ValueError("Jacobien non disponible pour calculer la matrice de covariance.")

    # Calcul des valeurs propres et vecteurs propres
    eigenvalues, eigenvectors = np.linalg.eig(covariance_matrix)

    # Longueurs des axes (pour un intervalle de confiance à 95%, multiplier par 2.4477)
    confidence_level = 2.4477  # Pour 95% de confiance (2 sigma)
    semi_axes_lengths = np.sqrt(eigenvalues) * confidence_level

    # Angle d'orientation du premier axe (en degrés)
    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    return semi_axes_lengths, angle

def plot_association_map_modified(results, stations_dict, title="Associations localisées"):
    """
    Carte montrant les positions des événements et des stations, avec ellipsoïdes d'erreur.
    """
    fig, ax = plt.subplots(figsize=(12, 8))

    # Extraire les positions des événements
    event_positions = []
    costs = []
    ellipses_data = []
    c=0
    lat_min_plot,lat_max_plot,lon_min_plot, lon_max_plot = np.float64(-32.22384054054054), np.float64(-31.123840540540606),np.float64(57.75945945945946), np.float64(58.859459459459394)
    for i, res in results.items():
        if hasattr(res, 'x') and len(res.x) >= 2:
            # Supposer que res.x = [t0, lat, lon] ou [lat, lon]
            if len(res.x) == 3:
                lat, lon = res.x[1], res.x[2]
                if lat_min_plot < lat < lat_max_plot :
                    if lon_min_plot < lon < lon_max_plot :
                        c+=1
            else:
                lat, lon = res.x[0], res.x[1]
            event_positions.append([lat, lon])
            cost = res.cost if hasattr(res, 'cost') else np.sum(res.fun**2)
            costs.append(cost)

            # Calculer l'ellipsoïde d'erreur
            try:
                semi_axes_lengths, angle = compute_ellipsoide_error(res)
                ellipses_data.append((lon, lat, semi_axes_lengths, angle))
            except Exception as e:
                print(f"Erreur pour l'événement {i}: {e}")
    print(c)
    if event_positions:
        event_positions = np.array(event_positions)
        costs = np.array(costs)

        # Scatter plot des événements colorés par cost
        scatter = ax.scatter(event_positions[:, 1], event_positions[:, 0],
                           c=costs, cmap='viridis_r', s=5, alpha=0.7,
                           label='Événements')
        plt.colorbar(scatter, label='Cost')

        # Ajouter les ellipsoïdes d'erreur
        # for lon, lat, semi_axes_lengths, angle in ellipses_data:
        #     ellipse = Ellipse(
        #         xy=(lon, lat),
        #         width=2 * semi_axes_lengths[0],
        #         height=2 * semi_axes_lengths[1],
        #         angle=angle,
        #         edgecolor='red',
        #         fc='None',
        #         lw=1,
        #         alpha=0.5
        #     )
        #     ax.add_patch(ellipse)

        # Ajouter les stations si disponibles
        if stations_dict:
            station_lats = []
            station_lons = []
            for station in stations_dict:
                pos = station.get_pos()
                station_lats.append(pos[0])
                station_lons.append(pos[1])

            ax.scatter(station_lons, station_lats, c='red', marker='^',
                      s=100, label='Stations', alpha=0.8)

    lat_min_plot,lat_max_plot,lon_min_plot, lon_max_plot = np.float64(-32.22384054054054), np.float64(-31.123840540540606),np.float64(57.75945945945946), np.float64(58.859459459459394)
    ax.set_xlim(lon_min_plot-0.2, lon_max_plot+0.2)
    ax.set_ylim(lat_min_plot-0.2, lat_max_plot+0.2)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

# Exemple d'utilisation
fig_mod = plot_association_map_modified(final_results, STATIONS)
plt.show()


## Drift analysis

In [ ]:
def compute_t_origin_uncertainty(res, sensors_positions, detection_times, velocities=None):
    """
    Calcule l'incertitude sur t_origin avec correction géographique.
    """
    min_date = np.argmin(detection_times)
    t0 = detection_times[min_date]

    # 1. Matrice de covariance des paramètres [lat, lon] (en degrés²)
    source_position = res.x[1:3]  # [latitude, longitude] en degrés
    J = res.jac

    n_obs = len(res.fun)
    n_params = 2  # latitude et longitude
    s_sq = (res.fun @ res.fun) / (n_obs - n_params)

    try:
        cov_deg = np.linalg.inv(J.T @ J) * s_sq  # Covariance en degrés²
    except np.linalg.LinAlgError:
        cov_deg = np.linalg.pinv(J.T @ J) * s_sq

    # 2. CONVERSION GÉOGRAPHIQUE : degrés → mètres
    lat_source = source_position[0]

    # Facteurs de conversion (mètres par degré)
    meters_per_deg_lat = 111320.0  # ≈ 111.32 km par degré de latitude
    meters_per_deg_lon = 111320.0 * np.cos(np.radians(lat_source))  # Dépend de la latitude

    # Matrice de transformation
    T = np.array([
        [meters_per_deg_lat, 0],
        [0, meters_per_deg_lon]
    ])

    # Conversion de la covariance : degrés² → mètres²
    cov_meters = T @ cov_deg @ T.T

    # 3. Calcul du gradient de t_origin par rapport à la position
    ref_sensor = sensors_positions[min_date]
    vector_to_source = np.array(source_position) - np.array(ref_sensor)

    # Conversion du vecteur en mètres pour le calcul de distance
    vector_to_source_meters = np.array([
        vector_to_source[0] * meters_per_deg_lat,
        vector_to_source[1] * meters_per_deg_lon
    ])

    distance_meters = np.linalg.norm(vector_to_source_meters)

    if distance_meters == 0:
        return float('inf')

    unit_vector = vector_to_source_meters / distance_meters

    # Vitesse du son
    if velocities is not None:
        velocity = velocities[min_date]
    else:
        velocity = SOUND_MODEL.get_sound_speed(source_position, ref_sensor, t0)

    if velocity == 0:
        return float('inf')

    # Gradient en mètres : ∂t_origin/∂pos = -unit_vector / velocity
    grad_t_origin = -unit_vector / velocity

    # 4. Propagation avec la covariance en mètres²
    variance_t_origin = grad_t_origin @ cov_meters @ grad_t_origin.T
    uncertainty_t_origin = np.sqrt(max(0, variance_t_origin))

    return uncertainty_t_origin

In [ ]:
import datetime
from itertools import compress
def process_for_drift(i, d, name=None):

    # ----- Préparation -----
    station_ids = associations[i][0][:, 0]
    det_ids     = associations[i][0][:, 1]
    n = len(station_ids)
#BASE
    # drift_ppm = {'ELAN':  np.float64(0),
    #              'H01W1': np.float64(0.0),
    #              'H04N1': np.float64(0.0),
    #              'H08S1': np.float64(0.0),
    #              'MADE':  np.float64(0.0),
    #              'MADW':  np.float64(0.0),
    #              'NEAMS': np.float64(0.0),
    #              'RTJ':   np.float64(0.0),
    #              'SSEIR': np.float64(0.0),
    #              'SSWIR': np.float64(0.0346),
    #              'SWAMS-bot': np.float64(0.0048),
    #              'WKER2': np.float64(0.0),
    #              'H04S1': np.float64(0.0)}
    #
    # offset = {'ELAN': np.float64(0),
    #           'H01W1': np.float64(0),
    #           'H04N1': np.float64(0),
    #           'H08S1': np.float64(0.0),
    #           'MADE': np.float64(0.0),
    #           'MADW': np.float64(0.0),
    #           'NEAMS': np.float64(0.0),
    #           'RTJ': np.float64(0.0),
    #           'SSEIR': np.float64(0.0),
    #           'SSWIR': np.float64(0.0),
    #           'SWAMS-bot': np.float64(0.0),
    #           'WKER2': np.float64(0.0),
    #           'H04S1': np.float64(0)}
# Iter1
#     drift_ppm = {'ELAN': np.float64(-0.142),
#                  'H01W1': np.float64(0.09328651963686112),
#                  'H04N1': np.float64(0.06760401079150478),
#                  'H04S1': np.float64(0.13684794474697048),
#                  'H08S1': np.float64(0.005488643341230808),
#                  'MADE': np.float64(0.06844502583246448),
#                  'MADW': np.float64(0.14386773304928813),
#                  'NEAMS': np.float64(0.19300636914410074),
#                  'RTJ': np.float64(0.14423783389259284),
#                  'SSEIR': np.float64(-0.0789466980247093),
#                  'SSWIR': np.float64(-0.026339959499136492),
#                  'SWAMS-bot': np.float64(0.0),
#                  'WKER2': np.float64(0.07580089411187704)}
# Iter2 :
    drift_ppm = {'ELAN': np.float64(-0.131710549912667),
             'H01W1': np.float64(0.08420817976058004),
             'H04N1': np.float64(0.005474507338212975),
             'H04S1': np.float64(0.04367580175236678),
             'H08S1': np.float64(0.0339800345578377),
             'MADE': np.float64(0.06750201481222673),
             'MADW': np.float64(0.13357977320875622),
             'NEAMS': np.float64(0.12430695166652732),
             'RTJ': np.float64(0.060539097101553196),
             'SSEIR': np.float64(-0.036162831789924375),
             'SSWIR': np.float64(0.02191309789424821),
             'SWAMS-bot': np.float64(0.0),
             'WKER2': np.float64(0.07208521358944064)}

    offset = {'ELAN': np.float64(-1.2),
              'H01W1': np.float64(0),
              'H04N1': np.float64(6.5),
              'H08S1': np.float64(0.0),
              'MADE': np.float64(0.0),
              'MADW': np.float64(0.0),
              'NEAMS': np.float64(0.0),
              'RTJ': np.float64(0.0),
              'SSEIR': np.float64(0.0),
              'SSWIR': np.float64(0.0),
              'SWAMS-bot': np.float64(0.0),
              'WKER2': np.float64(0.0),
              'H04S1': np.float64(6.5)}
    # timedelta(seconds=4.5)
    # Minimum 4 stations pour localiser ⇒ on doit pouvoir en retirer une
    if n < 5:
        return None

    # Noms, positions, temps, drift INITIAUX (non modifiés ensuite)
    names_all = [STATIONS[s].name for s in station_ids]
    pos_all   = [STATIONS[s].get_pos() for s in station_ids]

    det_all   = [IDX_TO_DET[d][0] for d in det_ids]
    # drift_errors = list(map(lambda j,k :timedelta(seconds=6.8)+timedelta(seconds=STATIONS[j].get_clock_error(k, drift_ppm=drift_mesured_10[STATIONS[j].name])) if STATIONS[j].name =="H04N1" else timedelta(seconds=STATIONS[j].get_clock_error(k, drift_ppm=drift_mesured_10[STATIONS[j].name])),associations[i][0][:,0], det_all))
    drift_errors = list(map(lambda j,k :timedelta(seconds=offset[STATIONS[j].name]) + timedelta(seconds=STATIONS[j].get_clock_error(k, drift_ppm=drift_ppm[STATIONS[j].name])),associations[i][0][:,0], det_all ))    #correction des drifts
    det_all = list(map(lambda d, e : d-e,det_all,drift_errors))
    det_all = list(map(lambda d : d.replace(tzinfo=timezone.utc),det_all))
    drift_all = [
        (0.1 + STATIONS[s].get_clock_error(IDX_TO_DET[d][0], drift_ppm=0.28)
         if "not_ok" in STATIONS[s].other_kwargs.values() or 'ok' in STATIONS[s].other_kwargs.values()
         else 0.1 if "H0" in STATIONS[s].name
         else 0.1)
        for s, d in zip(station_ids, det_ids)
    ]
    drift_all = np.abs(drift_all) / np.sqrt(3)

    # ----- Résultats leave-one-out -----
    pred_arr  = []
    obs_arr   = []
    tts       = []
    errors    = []
    uncertainty_t_origin = []
    x0 = final_results[i].x  # initialisation localisation

    for k in range(n):

        # Masque leave-one-out (on enlève SEULEMENT k)
        mask = [j != k for j in range(n)]

        pos_v   = list(compress(pos_all, mask))
        det_v   = list(compress(det_all, mask))
        drift_v = list(compress(drift_all, mask))

        # Distance max pour incertitudes
        max_idx = np.argmax(det_v)
        dmax = SOUND_MODEL.get_distance(x0[1:3], pos_v[max_idx])
        pick_unc = [2 + SOUND_MODEL.get_distance(x0[1:3], p)/dmax for p in pos_v]

        # ----- Localisation sans la station k -----
        res = SOUND_MODEL.localize_with_uncertainties(
            pos_v, det_v,
            y_min=lon_min-6, x_min=lat_min-6,
            y_max=lon_max+6, x_max=lat_max+6,
            drift_uncertainties=drift_v,
            pick_uncertainties=pick_unc,
            initial_pos=x0
        )

        # ----- Prédiction pour station k -----
        t0 = datetime.datetime.fromtimestamp(res.x[0], tz=timezone.utc)
        tt_k = timedelta(
            seconds=SOUND_MODEL.get_sound_travel_time(
                pos_all[k], res.x[1:3], t0
            )
        )
        pred_k = t0 + tt_k

        # Stockage
        tts.append(tt_k)
        pred_arr.append(pred_k)
        obs_arr.append(det_all[k])
        errors.append((pred_k - det_all[k]).total_seconds())
        uncertainty_t_origin.append(compute_t_origin_uncertainty(res,np.array(pos_v),np.array(det_v)))


    # ----- Construction finale -----
    d[i] = {
        'name': names_all,
        'pos': pos_all,
        'predicted_arrival': pred_arr,
        'observed_arrival': obs_arr,
        'travel_times': tts,
        'error': errors,
        'weight': uncertainty_t_origin  # à remplir si vous voulez un poids LOO moyen
    }


    return d


In [ ]:

import datetime as dt

df_test = pd.DataFrame.from_dict(final_results,orient='index')
df_test["weight"] = df_test["cost"]/df_test['fun'].apply(len)
df_test["time_origin"] = df_test['x'].apply(lambda x : pd.to_datetime(x[0],unit="s"))
df_test["lat"] = df_test['x'].apply(lambda x : x[1])
df_test["lon"] = df_test['x'].apply(lambda x : x[2])
d = dict()
st_name = None

def _wrap_process(i):
    return i, process_for_drift(i, {}, None)


def parallel_process_all(final_results):
    keys = list(final_results.keys())

    # IMPORTANT : limite les workers au nombre de CPU physiques
    n_workers = min( mp.cpu_count(), 28 )

    with mp.get_context("fork").Pool(n_workers) as pool:
        results = pool.map(_wrap_process, keys)

    # reconstruction du dictionnaire final
    d = {}
    for i, res in results:
        if res is not None:
            d[i] = res[i]
    return d

d = parallel_process_all(final_results)

# for i in tqdm(final_results.keys()):
#     _ = process_for_drift(i,d,st_name)


cols_to_explode = ['name','pos','predicted_arrival','observed_arrival','travel_times', 'error','weight']
df_drift = pd.DataFrame.from_dict(d, orient='index').explode(cols_to_explode)
df_drift_sta = df_drift[df_drift['name']==st_name]

In [ ]:
import seaborn as sns
plt.figure(figsize=(12,6))
sns.boxplot(
    data=df_drift,
    x='name',
    y='error',
    palette='tab20'
)

plt.xticks(rotation=45)
plt.ylabel('Erreur de prédiction (s)')
plt.title('Distribution des erreurs par station')
plt.grid(True, axis='y')
plt.show()


In [ ]:
drift_saved = {}
for s in STATIONS :
    st_name = s.name
    # for i in final_results.keys():
    #     _ = process_for_drift(i,d,st_name)
    df_drift = pd.DataFrame.from_dict(d, orient='index').explode(cols_to_explode)
    # df_drift = df_drift[(df_drift['predicted_arrival']<"2018-7-10") | (df_drift['predicted_arrival']>"2018-7-15")]
    df_drift =df_drift[df_drift["weight"]<30]
    df_drift_sta = df_drift[df_drift['name']==st_name]
# df_drift_sta = df_drift[df_drift['name']=="SSEIR"]


    # --- Données d'entrée ---
    x_abs = df_drift_sta['predicted_arrival'].values.astype(np.float64) / 1e9  # secondes
    y_abs = df_drift_sta['observed_arrival'].values.astype(np.float64) / 1e9   # secondes

    sigma =np.sqrt(2+df_drift_sta["weight"].values.astype(np.float64)**2)  # sigma en secondes

    if np.any(sigma <= 0):
        raise ValueError("sigma must be > 0 for all points")

    # --- Poids (pour pondérer les résidus) ---
    w = 1.0 / sigma           # w = 1/sigma
    w2 = w**2                 # w^2 = 1/sigma^2

    x0 = x_abs.min()
    x_rel_for_fit = x_abs - x0
    y_rel_for_fit = y_abs - x0


    # --- Modèle : y_rel = a * x_rel (vraiment sans offset) ---
    def residuals(params, x, y, w):
        a = params[0]

        return w * (y - a * x)

    # Régresser sur les données centrées
    res = least_squares(residuals, x0=[0.0], args=(x_rel_for_fit, y_rel_for_fit, w),loss='soft_l1',f_scale=0.5)

    a = res.x[0]

    # --- Estimation analytique de la variance de a ---
    # Var(a) = 1 / sum( x_i^2 / sigma_i^2 )
    JTJ_scalar = np.sum((x_abs**2) * w2)  # = sum( x_i^2 / sigma_i^2 )
    var_a_analytic = 1.0 / JTJ_scalar
    sigma_a_analytic = np.sqrt(var_a_analytic)

    # --- Résidus pondérés et chi-carré ---
    resid_weighted = residuals([a], x_abs, y_abs, w)
    chisq = np.sum(resid_weighted**2)
    N = x_abs.size
    p = 1  # nombre de paramètres
    dof = max(N - p, 1)
    reduced_chi2 = chisq / dof

    # CORRECTION 3: Covariance avec réduction du chi2
    # Cov(a) = sigma_a_analytic^2 * reduced_chi2
    cov_a = var_a_analytic * reduced_chi2
    sigma_a_jac = np.sqrt(cov_a)

    # Intervalle de confiance 95%
    ci_a = 1.96 * sigma_a_jac

    # --- Conversion en ppm ---
    drift_ppm = (a - 1) * 1e6
    ci_a_ppm = ci_a * 1e6

    print("\n" + "="*60)
    print(f"{st_name} Weighted Least Squares Regression Results (no offset, 95% CI)")
    print("="*60)
    print(f"Nombre de points: {N}")
    print(f"Drift factor (a): {a:.12f}")
    print(f"Drift (a - 1) in ppm: {drift_ppm:.3f} ppm")
    print(f"σ_a (analytique) = {sigma_a_analytic:.3e}  s^-1")
    print(f"σ_a (avec χ2 réduit) = {sigma_a_jac:.3e}  s^-1")
    print(f"95% CI (ppm) : [{drift_ppm - ci_a_ppm:.3f}, {drift_ppm + ci_a_ppm:.3f}]")
    print(f"χ2 = {chisq:.3f}, dof = {dof}, χ2_red = {reduced_chi2:.3f}")
    print("="*60 + "\n")

    # --- Préparation pour tracé ---
    x0 = x_abs.min()
    x_rel = x_abs - x0  # ancré au premier point

    drift_sec = y_abs - x_abs
    drift_us = drift_sec * 1e6
    e_drift_us = sigma * 1e6  # incertitude en μs

    drift_slope_us_per_s = (a - 1) * 1e6  # μs/s
    drift_fit_us = (a - 1) * x_rel * 1e6   # fit passant par l'origine (shifted)

    x_hours = x_rel / 3600.0
    drift_saved[st_name] = drift_ppm
    # --- Tracé ---
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.errorbar(x_hours, drift_us, yerr=e_drift_us, fmt='o', markersize=4,
                capsize=3, alpha=0.7, label='Observations (avec barres d\'erreur)')
    ax.plot(x_hours, drift_fit_us, 'r-', linewidth=2,
            label=f'Fit linéaire : slope = {drift_slope_us_per_s:.3f} μs/s ({drift_ppm:.1f} ppm)')
    ax.axhline(0.0, color='k', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('Temps depuis première arrivée (heures)', fontsize=11)
    ax.set_ylabel('Drift observé - attendu (μs)', fontsize=11)
    ax.set_title(f'{st_name} Estimation du Clock Drift Linéaire', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()
    plt.close(fig)



In [ ]:
df_drift = pd.DataFrame.from_dict(d, orient='index').explode(cols_to_explode)
df_drift = df_drift[(df_drift['predicted_arrival']<"2018-7-10") | (df_drift['predicted_arrival']>"2018-7-15")]
df_drift =df_drift[df_drift["weight"]<60]
st_name = 'ELAN'
df_drift_sta = df_drift[df_drift['name']==st_name]
# df_drift_sta = df_drift[df_drift['name']=="H04N1"]

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

# --- Données d'entrée ---
x_abs = df_drift_sta['predicted_arrival'].values.astype(np.float64) / 1e9  # secondes
y_abs = df_drift_sta['observed_arrival'].values.astype(np.float64) / 1e9   # secondes

sigma =np.sqrt(9+df_drift_sta["weight"].values.astype(np.float64)**2)  # sigma en secondes

if np.any(sigma <= 0):
    raise ValueError("sigma must be > 0 for all points")

# --- Poids (pour pondérer les résidus) ---
w = 1.0 / sigma           # w = 1/sigma
w2 = w**2                 # w^2 = 1/sigma^2

# --- Ancrer au premier point AVANT la régression ---
x0 = x_abs.min()
x_rel_for_fit = x_abs - x0  # temps relatif
y_rel_for_fit = y_abs - x0  # drift relatif au premier point

# --- Modèle : y_rel = a * x_rel (vraiment sans offset) ---
def residuals(params, x, y, w):
    a = params[0]
    b = params[1]

    return w * (y - a * x-b)

# Régresser sur les données centrées
res = least_squares(residuals, x0=[.0,-1], args=(x_rel_for_fit, y_rel_for_fit, w))

a = res.x[0]
b = res.x[1]

# --- Estimation analytique de la variance de a ---
# Var(a) = 1 / sum( x_i^2 / sigma_i^2 )
JTJ_scalar = np.sum((x_abs**2) * w2)  # = sum( x_i^2 / sigma_i^2 )
var_a_analytic = 1.0 / JTJ_scalar
sigma_a_analytic = np.sqrt(var_a_analytic)

# --- Résidus pondérés et chi-carré ---
resid_weighted = residuals([a,b], x_abs, y_abs, w)
chisq = np.sum(resid_weighted**2)
N = x_abs.size
p = 1  # nombre de paramètres
dof = max(N - p, 1)
reduced_chi2 = chisq / dof

# CORRECTION 3: Covariance avec réduction du chi2
# Cov(a) = sigma_a_analytic^2 * reduced_chi2
cov_a = var_a_analytic * reduced_chi2
sigma_a_jac = np.sqrt(cov_a)

# Intervalle de confiance 95%
ci_a = 1.96 * sigma_a_jac

# --- Conversion en ppm ---
drift_ppm = (a - 1) * 1e6
ci_a_ppm = ci_a * 1e6

print("\n" + "="*60)
print(f"{st_name} Weighted Least Squares Regression Results (no offset, 95% CI)")
print("="*60)
print(f"Nombre de points: {N}")
print(f"Drift factor (a): {a:.12f}")
print(f"Drift (a - 1) in ppm: {drift_ppm:.3f} ppm")
print(f"σ_a (analytique) = {sigma_a_analytic:.3e}  s^-1")
print(f"σ_a (avec χ2 réduit) = {sigma_a_jac:.3e}  s^-1")
print(f"95% CI (ppm) : [{drift_ppm - ci_a_ppm:.3f}, {drift_ppm + ci_a_ppm:.3f}]")
print(f"χ2 = {chisq:.3f}, dof = {dof}, χ2_red = {reduced_chi2:.3f}")
print("="*60 + "\n")

# --- Préparation pour tracé ---
x0 = x_abs.min()
x_rel = x_abs - x0  # ancré au premier point

drift_sec = y_abs - x_abs
drift_us = drift_sec * 1e6
e_drift_us = sigma * 1e6  # incertitude en μs

drift_slope_us_per_s = (a - 1) * 1e6  # μs/s
drift_fit_us = (a - 1) * x_rel * 1e6 +b*1e6  # fit passant par l'origine (shifted)

x_hours = x_rel / 3600.0

# --- Tracé ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(x_hours, drift_us, yerr=e_drift_us, fmt='o', markersize=4,
            capsize=3, alpha=0.7, label='Observations (avec barres d\'erreur)')
ax.plot(x_hours, drift_fit_us, 'r-', linewidth=2,
        label=f'Fit linéaire : slope = {drift_slope_us_per_s:.3f} μs/s ({drift_ppm:.1f} ppm)')
ax.axhline(0.0, color='k', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_xlabel('Temps depuis première arrivée (heures)', fontsize=11)
ax.set_ylabel('Drift observé - attendu (μs)', fontsize=11)
ax.set_title('Estimation du Clock Drift Linéaire', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


In [ ]:
b

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================================
# PRÉPARATION DES DONNÉES
# ============================================================================

cols = ['name', 'predicted_arrival', 'observed_arrival', 'weight']
df = (
    pd.DataFrame.from_dict(d, orient='index')
    .explode(cols)
    .reset_index()
    .rename(columns={'index': 'event_id'})
)

df = df[df['weight'] < 30].copy()

df['predicted_arrival'] = pd.to_datetime(df['predicted_arrival'])
df['observed_arrival'] = pd.to_datetime(df['observed_arrival'])
df['diff_sec'] = (df['observed_arrival'] - df['predicted_arrival']).dt.total_seconds()
df = df[(df['predicted_arrival']<"2018-7-10") | (df['predicted_arrival']>"2018-7-15")]
stations = sorted(df['name'].unique())
print(f"{len(df)} obs – {len(stations)} stations")

# ============================================================================
# START TIMES (timestamps en secondes)
# ============================================================================

start_times = {}
for s in STATIONS:
    start_times[s.name] = pd.Timestamp(s.date_start).timestamp()

print("\nTemps de démarrage:")
for s, t in sorted(start_times.items(), key=lambda x: x[1]):
    print(f"  {s:12s}: {pd.Timestamp(t, unit='s')}")

# ============================================================================
# TABLE WIDE
# ============================================================================

pivot_diff = df.pivot_table(index='event_id', columns='name', values='diff_sec').astype(float)
df['sigma_sq'] = df['weight'].astype(float)**2  # Variance picking
pivot_var = df.pivot_table(index='event_id', columns='name', values='sigma_sq').astype(float)

event_time = df.groupby('event_id')['predicted_arrival'].min()
t_event = event_time.view('int64').astype(np.float64) / 1e9  # Secondes Unix

# ============================================================================
# RÉGRESSIONS PAIRWISE AVEC START_TIMES
# ============================================================================

pairs = []

for i, si in enumerate(stations):
    for sj in stations[i+1:]:

        yi = pivot_diff[si]
        yj = pivot_diff[sj]
        vi = pivot_var[si]
        vj = pivot_var[sj]

        mask = yi.notna() & yj.notna() & vi.notna() & vj.notna()
        n = mask.sum()

        if n < 10:
            continue

        # Différence relative
        y = (yi[mask] - yj[mask]).values

        # Temps événement
        t = t_event.loc[mask].values

        t_centered = t - t.mean()
        # Variance
        sigma2 = (vi[mask] + vj[mask]).values
        w = 1.0 / sigma2

        # Régression WLS: y = slope*t + intercept
        # où slope = drift_i - drift_j
        # et intercept = drift_j*t_start_j - drift_i*t_start_i
        # X = np.vstack([t, np.ones(n)]).T
        # W = np.diag(w)
        #
        # XTW = X.T @ W
        # beta = np.linalg.solve(XTW @ X, XTW @ y)

        X = np.vstack([t_centered, np.ones(n)]).T
        W = np.diag(w)

        XTW = X.T @ W
        beta = np.linalg.solve(XTW @ X, XTW @ y)

        slope = beta[0]  # drift_i - drift_j
        intercept = beta[1]  # (offset_i + drift_i*t_mean) - (offset_j + drift_j*t_mean)


        # Résidus et chi2
        residuals = y - X @ beta
        sigma2_res = (residuals.T @ W @ residuals) / max(n - 2, 1)
        cov_beta = sigma2_res * np.linalg.inv(XTW @ X)

        slope = beta[0]
        slope_std = np.sqrt(cov_beta[0, 0])

        pairs.append({
            'station_i': si,
            'station_j': sj,
            'slope_ppm': slope * 1e6,
            'intercept_sec': intercept,
            'slope_std_ppm': slope_std * 1e6,
            'n': n
        })

df_pairs = pd.DataFrame(pairs)
print(f"\n{len(df_pairs)} paires valides")

# ============================================================================
# RECONSTRUCTION DRIFTS ABSOLUS
# ============================================================================

REF = 'SWAMS-bot'

idx = {s: i for i, s in enumerate(stations)}
# N = len(stations)
N_st = len(stations)
A_rows, b_vals, weights = [], [], []

# for _, r in df_pairs.iterrows():
#     i, j = idx[r.station_i], idx[r.station_j]
#
#     a = np.zeros(N)
#     a[i], a[j] = 1.0, -1.0
#
#     A_rows.append(a)
#     b_vals.append(r.slope_ppm)
#     weights.append(1.0 / (r.slope_std_ppm**2 + 1e-6))

for _, r in df_pairs.iterrows():
    i, j = idx[r.station_i], idx[r.station_j]

    # Equation 1: drift_i - drift_j = slope_ppm
    a_drift = np.zeros(2 * N_st)
    a_drift[i] = 1.0
    a_drift[j] = -1.0
    A_rows.append(a_drift)
    b_vals.append(r.slope_ppm)
    weights.append(1.0 / (r.slope_std_ppm**2 + 1e-6))

    # Equation 2: offset_i - offset_j = intercept
    a_offset = np.zeros(2 * N_st)
    a_offset[N_st + i] = 1.0
    a_offset[N_st + j] = -1.0
    A_rows.append(a_offset)
    b_vals.append(r.intercept_sec)
    weights.append(1.0 / 1.0)  # Adjust weight as needed

# # Contrainte REF = 0
# if REF in idx:
#     a = np.zeros(N)
#     a[idx[REF]] = 1.0
#     A_rows.append(a)
#     b_vals.append(0.0)
#     weights.append(1e6)

# Constraints
# 1. Reference drift = 0
a = np.zeros(2 * N_st)
a[idx[REF]] = 1.0
A_rows.append(a)
b_vals.append(0.0)
weights.append(1e6)

# 2. Reference offset = 0
a = np.zeros(2 * N_st)
a[N_st + idx[REF]] = 1.0
A_rows.append(a)
b_vals.append(0.0)
weights.append(1e6)

# 3. Fix offsets = 0 for all stations except ELAN and H04N1
stations_with_fixed_offset = [s for s in stations if s not in ['ELAN', 'H04N1', 'H04S1']]
for s in stations_with_fixed_offset:
    a = np.zeros(2 * N_st)
    a[N_st + idx[s]] = 1.0
    A_rows.append(a)
    b_vals.append(0.0)
    weights.append(1e6)  # Strong constraint

A = np.array(A_rows)
b = np.array(b_vals)
W = np.diag(weights)

ATA = A.T @ W @ A
ATb = A.T @ W @ b
x = np.linalg.solve(ATA, ATb)

residuals = A @ x - b
sigma2_post = (residuals.T @ W @ residuals) / max(len(b) - N, 1)
cov_x = sigma2_post * np.linalg.inv(ATA)
std_x = np.sqrt(np.maximum(0, np.diag(cov_x)))


drifts = x[:N_st]
offsets = x[N_st:]

df_drifts = pd.DataFrame({
    'drift_ppm': drifts,
    'drift_std_ppm': std_x[:N_st],
    'offset_sec': offsets
}, index=stations)

if REF in df_drifts.index:
    df_drifts.loc[REF] = 0.0

print("\nDrifts absolus (ppm)")
print(df_drifts.sort_values('drift_ppm'))

# ============================================================================
# VALIDATION
# ============================================================================

ground_truth = []
for s in STATIONS:
    ground_truth.append({
        'station': s.name,
        'true_drift_ppm': float(s.other_kwargs.get('clock_drift_ppm', 0.0))
    })
df_truth = pd.DataFrame(ground_truth).set_index('station')

df_comp = df_drifts.join(df_truth, how='inner')
df_comp['error_ppm'] = df_comp['drift_ppm'] - df_comp['true_drift_ppm']
df_comp['abs_error_ppm'] = df_comp['error_ppm'].abs()
df_comp['z_score'] = df_comp['error_ppm'] / df_comp['drift_std_ppm']
df_comp['within_2sigma'] = df_comp['z_score'].abs() <= 2

rmse = np.sqrt((df_comp['error_ppm']**2).mean())
mae = df_comp['abs_error_ppm'].mean()
bias = df_comp['error_ppm'].mean()
cov_2s = df_comp['within_2sigma'].mean()

print(f"\n{'='*60}")
print("VALIDATION")
print(f"{'='*60}")
print(f"RMSE          : {rmse:.3f} ppm")
print(f"MAE           : {mae:.3f} ppm")
print(f"Biais         : {bias:+.3f} ppm")
print(f"Couverture ±2σ: {cov_2s*100:.1f}%")

print(f"\nDÉTAIL:")
print(df_comp[['true_drift_ppm', 'drift_ppm', 'error_ppm', 'drift_std_ppm', 'z_score']].round(3).to_string())

# ============================================================================
# VISUALISATION
# ============================================================================

plt.figure()

# Plot 3: Drifts

colors = ['red' if s == REF else 'steelblue' for s in df_drifts.index]
plt.bar(range(len(df_drifts)), df_drifts['drift_ppm'], color=colors, alpha=0.7)
plt.errorbar(range(len(df_drifts)), df_drifts['drift_ppm'],
           yerr=df_drifts['drift_std_ppm'], fmt='none', ecolor='gray', capsize=5)
plt.axhline(0, color='k', linestyle='--')
ax = plt.gca()
ax.set_xticks(range(len(df_drifts)))
ax.set_xticklabels(df_drifts.index, rotation=45, ha='right')
ax.set_ylabel('Drift (ppm)')
ax.set_title(f'Drifts absolus (Réf: {REF})')
ax.grid(axis='y', alpha=0.3)

#
print(f"\n✅ Terminé")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Reference station
REF = 'SWAMS-bot'
event_time = df.groupby('event_id')['predicted_arrival'].min()
t_event = event_time.view('int64').astype(np.float64) / 1e9
# Get stations excluding reference
stations_to_plot = [s for s in stations if s != REF]

# Create subplot grid
n_stations = len(stations_to_plot)
n_cols = 3
n_rows = int(np.ceil(n_stations / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

# Plot each station vs reference
for idx, si in enumerate(stations_to_plot):
    ax = axes[idx]

    yi = pivot_diff[si]
    yj = pivot_diff[REF]
    vi = pivot_var[si]
    vj = pivot_var[REF]

    mask = yi.notna() & yj.notna() & vi.notna() & vj.notna()
    n = mask.sum()

    if n < 10:
        continue

    # Get common indices
    common_idx = yi[mask].index

    # Extract values using common indices
    y = (yi[common_idx] - yj[common_idx]).values
    t = t_event.loc[common_idx].values
    sigma2 = (vi[common_idx] + vj[common_idx]).values
    w = 1.0 / sigma2

    # Now all arrays match
    n_points = len(y)
    X = np.vstack([t, np.ones(n_points)]).T
    W = np.diag(w)
    beta = np.linalg.solve(X.T @ W @ X, X.T @ W @ y)

    # Fit line
    tt = np.linspace(t.min(), t.max(), 200)
    yy = beta[0] * tt + beta[1]

    # Convert to days from first event
    t_days = (t - t.min()) / 86400
    tt_days = (tt - t.min()) / 86400

    # Get drift from results
    drift_ppm = df_drifts.loc[si, 'drift_ppm']
    offset = df_drifts.loc[si, 'offset_sec']

    # Plot
    scatter = ax.scatter(t_days, y, c=w, cmap='viridis', alpha=0.6, s=20)
    ax.plot(tt_days, yy, 'r-', lw=2,
            label=f'drift={drift_ppm:.3f} ppm\noffset={offset:.2f} s')

    ax.axhline(0, color='gray', ls='--', alpha=0.3)
    ax.set_xlabel('Days from first event')
    ax.set_ylabel(f'Δt (s)')
    ax.set_title(f'{si} vs {REF}', fontweight='bold')
    ax.legend(fontsize=8, loc='best')
    ax.grid(alpha=0.3)

# Remove empty subplots
for idx in range(n_stations, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.show()

# Also create a single detailed plot for a specific pair if needed
def plot_pair_drift_detailed(si, sj=REF):
    """Plot detailed pairwise drift with weights colorbar"""
    yi = pivot_diff[si]
    yj = pivot_diff[sj]
    vi = pivot_var[si]
    vj = pivot_var[sj]

    mask = yi.notna() & yj.notna() & vi.notna() & vj.notna()
    if mask.sum() < 10:
        print(f"Insufficient data for {si} vs {sj}")
        return

    y = (yi[mask] - yj[mask]).values
    t = t_event.loc[mask].values
    sigma2 = (vi[mask] + vj[mask]).values
    w = 1.0 / sigma2

    # WLS regression
    n_points = len(t)
    X = np.vstack([t, np.ones(n_points)]).T
    W = np.diag(w)
    beta = np.linalg.solve(X.T @ W @ X, X.T @ W @ y)

    # Fit line
    tt = np.linspace(t.min(), t.max(), 200)
    yy = beta[0] * tt + beta[1]

    # Convert to days
    t_days = (t - t.min()) / 86400
    tt_days = (tt - t.min()) / 86400

    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(t_days, y, c=w, cmap='viridis', alpha=0.7, s=30)
    plt.plot(tt_days, yy, 'r-', lw=2.5,
             label=f"slope = {beta[0]*1e6:.3f} ppm\nintercept = {beta[1]:.3f} s")
    plt.colorbar(scatter, label='Weight (1/σ²)')
    plt.axhline(0, color='gray', ls='--', alpha=0.3)
    plt.xlabel("Days from first event", fontsize=12)
    plt.ylabel(f"Δt {si} − {sj} (s)", fontsize=12)
    plt.title(f"Pairwise Drift: {si} vs {sj}", fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

# Example: plot detailed view for H04N1
plot_pair_drift_detailed('H04S1')
plot_pair_drift_detailed('ELAN')

In [ ]:
dict(df_drifts.offset_sec)

In [ ]:
dict(df_drifts.drift_ppm)

In [ ]:
it1 = {'ELAN': np.float64(-0.142),
             'H01W1': np.float64(0.09328651963686112),
             'H04N1': np.float64(0.06760401079150478),
             'H04S1': np.float64(0.13684794474697048),
             'H08S1': np.float64(0.005488643341230808),
             'MADE': np.float64(0.06844502583246448),
             'MADW': np.float64(0.14386773304928813),
             'NEAMS': np.float64(0.19300636914410074),
             'RTJ': np.float64(0.14423783389259284),
             'SSEIR': np.float64(-0.0789466980247093),
             'SSWIR': np.float64(-0.026339959499136492),
             'SWAMS-bot': np.float64(0.0),
             'WKER2': np.float64(0.07580089411187704)}


it2 ={'ELAN': np.float64(0.010289450087332992),
 'H01W1': np.float64(-0.009078339876281087),
 'H04N1': np.float64(-0.06212950345329181),
 'H04S1': np.float64(-0.0931721429946037),
 'H08S1': np.float64(0.02849139121660689),
 'MADE': np.float64(-0.0009430110202377426),
 'MADW': np.float64(-0.010287959840531918),
 'NEAMS': np.float64(-0.06869941747757342),
 'RTJ': np.float64(-0.08369873679103965),
 'SSEIR': np.float64(0.042783866234784924),
 'SSWIR': np.float64(0.0482530573933847),
 'SWAMS-bot': np.float64(0.0),
 'WKER2': np.float64(-0.003715680522436401)}

new = it1

for i in new.keys():
    new[i]+=it2[i]

In [ ]:
new

## Compute SL

In [ ]:
from utils.data_reading.sound_data.sound_file_manager import DatFilesManager, WFilesManager
from concurrent.futures import ProcessPoolExecutor, as_completed

# Root directory containing subfolders for each station
DATA_ROOT = "/media/rsafran/CORSAIR/OHASISBIO"
# Subfolder (e.g. year) within DATA_ROOT
DATASET   = "OHASISBIO-2018"


#exemple pour aller chercher les détéctions times d'une station
def process(i,RL=False):
    station = list(map(lambda j: STATIONS[j], associations[i][0][:,0]))

    # if len(station) < 7:
    #     return None

    det = list(map(lambda j: IDX_TO_DET[j][0], associations[i][0][:,1]))
    if RL :
        RL = list(map(lambda k : IDX_TO_RL[k][0], associations[i][0][:,1]))
        NL = list(map(lambda k : IDX_TO_RL[k][1], associations[i][0][:,1]))
        return station,det, RL, NL

    return(station, det)

def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a


def bandpass_filter(data, lowcut, highcut, fs):
    b, a = butter_bandpass(lowcut, highcut, fs)
    y = filtfilt(b, a, data)
    return y

def refine_single_detection(station_obj, det_time, lowcut=None, highcut=None):
    """
    Refine one pick time by locating the max energy within ±SMOOTH_WINDOW_SEC.
    Also computes a simple SNR ratio.
    Returns a dict or None on failure.
    """
    # load waveform ±3 minutes
    start = det_time - timedelta(minutes=5)
    end   = det_time + timedelta(minutes=5)
    if station_obj.dataset=="OHASISBIO-2018":
        raw = 'raw'
        mgr = DatFilesManager(f"{DATA_ROOT}/{DATASET}/{station_obj.name}", kwargs=raw)
        data = mgr.get_segment(start, end)
        sampling_f = mgr.sampling_f
    else :
        mgr = WFilesManager(f"/media/rsafran/CORSAIR/CTBT/CTBTO_2018/{station_obj.name}_2018/")
        data = mgr.get_segment(start, end)
        sampling_f = round(mgr.sampling_f)

    n = len(data)
    t = np.arange(n) / sampling_f - (det_time - start).total_seconds()
    if lowcut is not None and highcut is not None:
        data = bandpass_filter(data,lowcut, highcut, round(sampling_f))

    return data,t, sampling_f


def compute_RL(data, fs, window_sec=10, step_sec=1, signal_margin_sec=5):

    P_REF = 1e-6

    # -------------------------
    # Fenêtres glissantes
    # -------------------------
    win = int(window_sec * fs)
    step = int(step_sec * fs)
    nwin = int((len(data) - win) / step) + 1

    time_centers = np.zeros(nwin)
    SPL_RMS = np.zeros(nwin)

    for i in range(nwin):
        start = i * step
        end = start + win
        window = data[start:end]
        time_centers[i] = (start) / fs

        # --- RMS ---
        p_rms = np.sqrt(np.mean(window**2))
        SPL_RMS[i] = 20 * np.log10(p_rms / P_REF)

    # -------------------------
    # Identification des zones (asymétrique)
    # -------------------------
    total_duration = len(data) / fs
    signal_start_time = total_duration/2  # Le signal COMMENCE au début


    # Zone étendue avec marges de sécurité
    signal_zone_start = signal_start_time - signal_margin_sec
    signal_zone_end = signal_start_time + window_sec+signal_margin_sec

    # Masques booléens
    is_signal = (time_centers >= signal_zone_start) & (time_centers <= signal_zone_end)
    is_noise = ~is_signal

    # -------------------------
    # Extraction des niveaux
    # -------------------------
    # SPL maximum dans la zone du signal
    if np.any(is_signal):
        SPL_signal_max = np.max(SPL_RMS[is_signal])
    else:
        SPL_signal_max = np.nan
        print("Attention : aucune fenêtre dans la zone du signal")

    # Niveau de bruit (médiane hors signal)
    if np.any(is_noise):
        noise_level = np.median(SPL_RMS[is_noise])
    else:
        noise_level = np.nan
        print("Attention : aucune fenêtre de bruit disponible")

    return SPL_signal_max, noise_level


# -------------------------------------------------------
# 1. Mapping inverse : idx_det → station
# -------------------------------------------------------
IDX_TO_STATION = {}
for s in STATIONS:
    for det in DETECTIONS[s]:
        idx_det = int(det[-1])          # dernier champ = index
        IDX_TO_STATION[idx_det] = s

# -------------------------------------------------------
# 2. Paramètres du filtre (adaptez selon vos données)
# -------------------------------------------------------
LOWCUT  = 5.0    # Hz
HIGHCUT = 80.0   # Hz
N_WORKERS = 20

# -------------------------------------------------------
# Fonction top-level (obligatoire pour pickle avec multiprocessing)
# -------------------------------------------------------
def process_one(args):
    idx_det, det, station, lowcut, highcut = args
    det_time = det[0]
    try:
        data, t, fs = refine_single_detection(station, det_time, lowcut=lowcut, highcut=highcut)
        spl_max, noise_level = compute_RL(data, fs)
        return idx_det, (spl_max, noise_level)
    except Exception as e:
        return idx_det, (np.nan, np.nan)

# -------------------------------------------------------
# Préparation des arguments (pas de globaux dans les workers)
# -------------------------------------------------------
det_idx = list(map(lambda k : list(associations[k][0][:,1]), final_results.keys()))

def flatten(xss):
    return [x for xs in xss for x in xs]

det_idx = flatten(det_idx)

args_list = [
    (idx, IDX_TO_DET[idx], IDX_TO_STATION[idx], LOWCUT, HIGHCUT)
    for idx in det_idx
]

args_list=args_list
# -------------------------------------------------------
# Exécution
# -------------------------------------------------------
IDX_TO_RL   = {}
FAILED_IDXS = []

with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
    futures = {executor.submit(process_one, args): args[0] for args in args_list}
    with tqdm(total=len(futures), desc="Computing RL") as pbar:
        for future in as_completed(futures):
            idx_det, result = future.result()
            IDX_TO_RL[idx_det] = result
            if np.isnan(result[0]):
                FAILED_IDXS.append(idx_det)
            pbar.update(1)

print(f"Terminé. {len(IDX_TO_RL)} entrées, {len(FAILED_IDXS)} échecs.")

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, welch
from datetime import timedelta


def build_rl_dataframe(associations, event_list=None):
    """
    associations : dict-like mapping event_id -> list of station entries.
      - On gère deux formats usuels :
         1) associations[event_id] = [(station_obj, det_time), (station_obj2, det_time2), ...]
         2) associations[event_id] = [(station_name_or_id, det_time), ...]
      Si le format n'est pas celui-ci, on tente une retombée sur process(event_id) (si disponible).
    event_list : optionnel, liste d'event_id à traiter (par défaut toutes les clés de associations)
    Retour : pandas.DataFrame
    """
    rows = []
    for association_id in tqdm(event_list):
        asso_data = process(association_id,RL=True)
        for i in range(len(asso_data[0])):
            station_obj = asso_data[0][i]
            det_time = asso_data[1][i]
            RL = asso_data[2][i]
            NL = asso_data[3][i]
            station_id = associations[association_id][0][i,0]

            rows.append({
                'event_id': association_id,
                'station_id': station_id,
                'station_name': getattr(station_obj, 'name', None),
                'det_time': det_time,
                'SPL_RMS_dB' : RL,
                'Noise_level' : NL,
                })

    df = pd.DataFrame(rows)
    return df

df = build_rl_dataframe(associations,event_list=list(final_results.keys()))
print(df.head())


In [ ]:
from pyproj import Geod

bathy = {'NEAMS'   : 3840,
'MADW'     : 4848 ,
'SSWIR'    : 4390,
'MADE'     : 4018,
'SSEIR'    : 4180,
'RTJ'      : 3589,
'WKER2'    : 4536,
'H04N1'    : 548,
'ELAN'     : 4820 ,
'H08S1'    :1300 ,
'H01W1'    :1063.0  ,
'SWAMS-bot': 3433,
         'H04S1':550}

geod = Geod(ellps="WGS84")  # Utilise l'ellipsoïde WGS84
distances_df = []
for i in final_results.keys():
    names = list(map(lambda j: STATIONS[j].name, associations[i][0][:,0]))
    stations = list(map(lambda j: STATIONS[j].get_pos(include_depth=True), associations[i][0][:,0])) #lat lon stations
    stations = np.array(stations)
    vertical_dist = list(map(lambda j: bathy[STATIONS[j].name] - STATIONS[j].get_pos(include_depth=True)[2], associations[i][0][:,0]))
    pos = final_results[i].x[1::] #lat lon final event
    pos = np.array([pos]*len(stations))
    _, _, distances = geod.inv(pos[:,1], pos[:,0], stations[:,1], stations[:,0]) #lon lat
    distances_df.append({'event_id': i,
                         'station_name': names,
                         'vertical_dist': vertical_dist,
                         'distances': distances})

distances_df = pd.DataFrame(distances_df)
distances_df = distances_df.explode(['distances','vertical_dist',"station_name"])
df = df.merge(distances_df, how='left', on=['event_id','station_name'])

In [ ]:
df['distance_km'] = df['distances'] / 1000.0
df = df[df['distance_km'] > 1]  # éviter zone très proche
df['log10_r'] = df['distances'].apply(lambda x: np.log10(x))
df['20log10_rv'] = df['vertical_dist'].apply(lambda x: 20*np.log10(x))


In [ ]:
df.loc[df['Noise_level']<0,["Noise_level"]]=200
df["SPL_RMS_dB_ns"]=20*np.log10(10**(df["SPL_RMS_dB"]/20)+
                                 10**(df['Noise_level']/20))
df["SPL_RMS_dB_nsv"]= 20*np.log10(10**(df["SPL_RMS_dB"]/20)+
                                 10**(df['Noise_level']/20)) + df['20log10_rv']
df["SNR_dB"]= df["SPL_RMS_dB"]-df['Noise_level']

df["SPL_RMS_dB_v"]= df["SPL_RMS_dB"]+0.5*df['20log10_rv']

df['SL']= df['SPL_RMS_dB']+10*df['log10_r']+0.5*df["20log10_rv"]
# Replacing infinite with nan
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Dropping all the rows with nan values
df.dropna(inplace=True)

# Printing df
df

In [ ]:
RL_COL = 'SPL_RMS_dB_v'  # recommandé 'SPL_RMS_dB', 'SEL_dB', 'Dziak_PSD_mean_dB','BandPower_dB', 'SPL_PP_dB', 'TPEF_dB', 'PSD_P90_dB', 'F_dom_Hz','F_centroid_Hz', 'TercileBandPower_dB',


In [ ]:
from scipy.stats import linregress
import statsmodels.api as sm
results = []

for ev in df['event_id'].unique():
    subset = df[df['event_id'] == ev].dropna(subset=[RL_COL])

    if len(subset) < 5:
        continue

    x = subset['log10_r'].values
    y = subset[RL_COL].values
    w=subset['SNR_dB_v'].values
    w=np.clip(w,0.1,None)
    # slope, intercept, r_value, p_value, std_err = linregress(x, y)
    #


    X = sm.add_constant(x)
    wls = sm.WLS(y, X, weights=w).fit()

    slope = wls.params[1]
    intercept = wls.params[0]
    r_value = np.sqrt(wls.rsquared)
    # print(wls.summary())

    results.append({
        'event_id': ev,
        'n_stations': len(subset),
        'intercept': intercept,
        'slope_dB_per_decade': slope,
        'R2': r_value**2
    })

summary_df = pd.DataFrame(results)
print(summary_df.describe())


plt.figure(figsize=(8,5))
plt.hist(summary_df['slope_dB_per_decade'], bins=20)
plt.axvline(-10, color='red', linestyle='--', label='Cylindrique')
plt.axvline(-20, color='green', linestyle='--', label='Sphérique')
plt.xlabel("Slope (dB per decade)")
plt.ylabel("Count")
plt.legend()
plt.title(f"Distribution des pentes RL vs distance {RL_COL}")
plt.show()


# RL_COL = 'BandPower_dB'
plt.figure(figsize=(8,6))

df['RL_centered'] = (
    df[RL_COL] - df.groupby('event_id')[RL_COL].transform('median')
    # - df["Noise_level"] #df.groupby('event_id')[RL_COL].transform('median')
)
# Utilise l'intercept de la régression comme référence (RL à 1 km)
# df['RL_centered'] = df[RL_COL] - df['event_id'].map(
#     summary_df.set_index('event_id')['intercept']  # à ajouter dans results
# )

# for ev in df['event_id'].unique():
#     subset = df[df['event_id']==ev]
#     plt.scatter(subset['distance_km'], subset['RL_centered'],c=subset['station_id'],cmap= cm,norm=norm,alpha=0.8,)

# Créer une colormap personnalisée
unique_stations = sorted(df['station_id'].unique())
n_stations = len(unique_stations)
palette = cc.glasbey[:n_stations]
cmap = ListedColormap(palette)

#Plot result
s_min, s_max = 1, 150  # taille en points^2, à ajuster
snr = df['SNR_dB']
snr_norm = 1-(snr - snr.min()) / (snr.max() - snr.min())
# s_min, s_max = 2, 150
s = s_min * np.exp(snr_norm * np.log(s_max / s_min))

scatter = plt.scatter(df['distance_km'],df['RL_centered'], c=df['station_id'],cmap=cmap,s=s ,alpha=0.25)

ax = plt.gca()
# Inverse de : s = s_min * exp(snr_norm * log(s_max/s_min))
# avec snr_norm = 1 - (snr - snr.min()) / (snr.max() - snr.min())
kw = dict(
    prop="sizes",
    num=5,
    color="gray",
    fmt="{x:.0f} dB",
    func=lambda s: snr.max() - (np.log(s / s_min) / np.log(s_max / s_min)) * (snr.max() - snr.min())
)

legend2 = ax.legend(*scatter.legend_elements(**kw),
                    loc="upper right", title="SNR")


#COLORBAR
boundaries = np.arange(-0.5, n_stations + 0.5, 1)
cbar = plt.colorbar(scatter, boundaries=boundaries, ticks=range(n_stations))
station_dict = dict(zip(df['station_id'], df['station_name']))
cbar.ax.set_yticklabels([station_dict[station_id] for station_id in unique_stations])
cbar.set_label('Stations')
cbar.solids.set(alpha=1)
plt.xscale('log')
# plt.ylim(-20,20)
plt.xlabel("Distance (km)")
plt.ylabel(f"{RL_COL} normalisé (dB)")
plt.title(f"Décroissance normalisée du {RL_COL}")
plt.grid()
plt.show()

w=df['SNR_dB_v'].values
w=np.clip(w,0.1,None)
x =df['log10_r'].values
y = df['RL_centered'].values

# slope, intercept, r_value, p_value, std_err = linregress(x, y)
X = sm.add_constant(x)
wls = sm.WLS(y, X, weights=w).fit()

slope = wls.params[1]
r_value = np.sqrt(wls.rsquared)

print(f"Pente globale : {slope:.2f} dB/decade")
print(f"R² global : {r_value**2:.2f}")


In [ ]:
#NV
#            event_id   n_stations  slope_dB_per_decade           R2
# count  2.095000e+03  2095.000000          2095.000000  2095.000000
# mean   5.723616e+05     6.757518            -9.273943     0.333032
# std    4.985793e+05     1.095885             8.282586     0.251948
# min    9.000000e+01     4.000000           -52.891325     0.000001
# 25%    1.727230e+05     6.000000           -13.891677     0.105975
# 50%    4.279150e+05     7.000000            -9.194358     0.300208
# 75%    8.668550e+05     8.000000            -4.373797     0.531780
# max    2.018459e+06     9.000000            33.613658     0.996064

#N
#            event_id   n_stations  slope_dB_per_decade            R2
# count  2.095000e+03  2095.000000          2095.000000  2.095000e+03
# mean   5.723616e+05     6.757518           -10.396424  4.074645e-01
# std    4.985793e+05     1.095885             8.511783  2.806673e-01
# min    9.000000e+01     4.000000           -57.138779  1.356918e-07
# 25%    1.727230e+05     6.000000           -14.906586  1.496413e-01
# 50%    4.279150e+05     7.000000           -10.252901  3.986794e-01
# 75%    8.668550e+05     8.000000            -5.105918  6.467671e-01
# max    2.018459e+06     9.000000            29.933708  9.930430e-01

#rien
#            event_id   n_stations  slope_dB_per_decade            R2
# count  2.095000e+03  2095.000000          2095.000000  2.095000e+03
# mean   5.723616e+05     6.757518            -5.795265  2.127829e-01
# std    4.985793e+05     1.095885             7.271607  2.111598e-01
# min    9.000000e+01     4.000000           -60.391366  2.622232e-07
# 25%    1.727230e+05     6.000000            -9.967825  3.684074e-02
# 50%    4.279150e+05     7.000000            -6.023303  1.464192e-01
# 75%    8.668550e+05     8.000000            -1.440653  3.267511e-01
# max    2.018459e+06     9.000000            27.115688  9.471931e-01


In [ ]:

Envent_RL = df.groupby('event_id')['SL'].median()

In [ ]:


quality = []
rows = []
for i in list(final_results):
    res = final_results[i]
    rows.append({
        "event_id":i,
        "lat": res.x[1],
        "lon": res.x[2],
        "origin_date": res.x[0],
        "cost": res.cost
    })
    # quality.append(analyze_residuals(res))+

# quality = pd.DataFrame(quality)
df_event = pd.DataFrame(rows)
df_event["origin_date"] = pd.to_datetime(df_event["origin_date"], unit="s",utc=True)
df_event = df_event.merge(Envent_RL,on='event_id')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

selection = df_event #[(df_event['origin_date'] > "2018-07-10") & (df_event['origin_date'] < "2018-07-15")]

# Make sure origin_date is datetime
selection = selection.copy()
selection['origin_date'] = pd.to_datetime(selection['origin_date'])

fig, ax = plt.subplots()
ax.plot(selection["origin_date"], selection['SL'], '+', color='red')

# Format x-axis to show dates nicely
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())

plt.gcf().autofmt_xdate()  # rotate labels to avoid overlap
plt.tight_layout()
plt.show()

In [ ]:
# PATH = '/media/rsafran/CORSAIR/isc-gem/'
PATH = '/media/rsafran/CORSAIR/isc-ehb/'
# NAME_ISC = 'isc-bulletin_swir_2018.csv'
NAME_ISC = 'isc-ehb.csv'
# NAME_ISC = 'isc-gem-cat.csv'
# NAME_ISC = 'USGS_cat.csv' #in ehb file
isc_cnames = ['date','lat','lon','smajax','sminax','strike','q','depth','unc','q.1','mw','unc','q.2','s','mo','fac','mo_auth','mpp','mpr','mrr','mrt','mtp','mtt','str1','dip1','rake1','str2','dip2','rake2','type','eventid']

isc_cat_csv = pd.read_csv(PATH + NAME_ISC, comment='#',header=0,skipinitialspace=True)
# isc_cat_csv['date'] = pd.to_datetime(isc_cat_csv['date'],format='ISO8601', utc=True) #gem
# isc_cat_csv['date'] = pd.to_datetime(isc_cat_csv['time'],format='ISO8601', utc=True) #USGS
isc_cat_csv.loc[:,'date'] = pd.to_datetime(isc_cat_csv.loc[:,'DATE'] + ' ' + isc_cat_csv.loc[:,'TIME'], format='ISO8601', utc=True) #ehb

isc_cat_csv.dropna(axis=1, how='any', inplace=True)
isc_cat_csv.rename(columns= {'LAT':'lat', 'LON':'lon','latitude':'lat', 'longitude':'lon', "DEPTH": "depth","MAG":'mw',"mag":'mw', 'EVENTID':'eventid', 'id':'eventid'}, inplace=True)
isc_cat_csv.sort_values('date', inplace=True)
# isc_cat_csv = isc_cat_csv[(isc_cat_csv['date'] > "2018-07-10") & (isc_cat_csv['date'] < "2018-07-15" )]
#
# (lat_min, lat_max),(lon_min, lon_max) =(np.float64(-32.22384054054054), np.float64(-31.123840540540606)) ,(np.float64(57.75945945945946), np.float64(58.859459459459394))
# isc_cat_csv = isc_cat_csv[(isc_cat_csv['lat'] > lat_min) & (isc_cat_csv['lat'] < lat_max) & (isc_cat_csv['lon'] > lon_min) & (isc_cat_csv['lon'] < lon_max) ]

In [ ]:
isc_cat_csv.sort_values('date',inplace=True)
selection.sort_values('origin_date', inplace=True)
tmp = pd.merge_asof(
    isc_cat_csv,
    selection,
    left_on='date',
    right_on='origin_date',
    tolerance=pd.Timedelta(seconds=180),
    direction='nearest',

)

In [ ]:
tmp.dropna(inplace=True)
print(len(tmp))

In [ ]:
tmp.plot('SL','mw',style='+')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Compute cumulative frequency ---
magnitudes = df_event['SL'].dropna().values  # use your magnitude column

mag_bins = np.arange(magnitudes.min(), magnitudes.max(), 0.1)
cumulative_count = np.array([np.sum(magnitudes >= m) for m in mag_bins])

# Remove zeros (log undefined)
mask = cumulative_count > 0
mag_bins = mag_bins[mask]
cumulative_count = cumulative_count[mask]

# --- 2. Fit a/b values (linear regression on log10) ---
log_N = np.log10(cumulative_count)
coeffs = np.polyfit(mag_bins, log_N, 1)  # coeffs[0]=b, coeffs[1]=a
b_value = -coeffs[0]
a_value = coeffs[1]

fit_line = np.polyval(coeffs, mag_bins)

# --- 3. Plot ---
fig, ax = plt.subplots()
ax.scatter(mag_bins, cumulative_count, color='red', marker='+', label='Observed')
ax.plot(mag_bins, 10**fit_line, color='blue', label=f'G-R fit: b={b_value:.2f}, a={a_value:.2f}')
plt.yscale('log')
ax.set_xlabel('Magnitude (M)')
ax.set_ylabel('log₁₀ N (cumulative)')
ax.set_title('Gutenberg-Richter Relation')
ax.legend()
plt.tight_layout()
plt.grid()
plt.show()

print(f"a = {a_value:.3f}")
print(f"b = {b_value:.3f}")